In [ ]:
# ============================================================
# REVIEWER DATA AUDIT 1
# Exact instruments, CRSP total returns, sample feasibility
# Run this entire cell in WRDS JupyterLab.
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import wrds
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

# ------------------------------------------------------------
# 1. Frozen candidate universe
# ------------------------------------------------------------

ASSETS_BASE_METALS = ["DBE", "GLD", "DBA", "DBB"]
ASSETS_COPPER      = ["DBE", "GLD", "DBA", "CPER"]
BENCHMARKS         = ["GSG", "DBC"]
ALL_TICKERS        = sorted(
    set(ASSETS_BASE_METALS + ASSETS_COPPER + BENCHMARKS)
)

TRAIN_END = pd.Timestamp("2018-12-31")
OOS_START = pd.Timestamp("2019-01-31")
LOOKBACK_MONTHS = 12

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTDIR = Path.home() / f"gsci_reviewer_audit_{RUN_ID}"
OUTDIR.mkdir(parents=True, exist_ok=False)

print("=" * 100)
print("REVIEWER DATA AUDIT")
print("=" * 100)
print("Output directory:", OUTDIR)
print("Tickers:", ALL_TICKERS)
print("Train end:", TRAIN_END.date())
print("OOS start:", OOS_START.date())
print()

# ------------------------------------------------------------
# 2. Connect to WRDS
# ------------------------------------------------------------

print("Connecting to WRDS...")
db = wrds.Connection()
print("CONNECTED")
print()

# ------------------------------------------------------------
# 3. Discover available CRSP monthly table
# ------------------------------------------------------------

schema_sql = """
SELECT
    lower(table_name) AS table_name,
    lower(column_name) AS column_name
FROM information_schema.columns
WHERE table_schema = 'crsp'
  AND lower(table_name) IN (
      'msf',
      'msf_v2',
      'stocknames'
  )
ORDER BY table_name, ordinal_position
"""

schema = db.raw_sql(schema_sql)
schema.to_csv(OUTDIR / "crsp_schema_discovery.csv", index=False)

available = (
    schema.groupby("table_name")["column_name"]
    .apply(set)
    .to_dict()
)

print("Available relevant CRSP tables:")
for table, columns in available.items():
    print(f"  {table}: {len(columns)} columns")

if "stocknames" not in available:
    raise RuntimeError(
        "crsp.stocknames was not found. "
        "Stop and paste the table list printed above."
    )

if (
    "msf_v2" in available
    and {"permno", "mthcaldt", "mret"}.issubset(available["msf_v2"])
):
    RETURN_TABLE = "crsp.msf_v2"
    DATE_COLUMN = "mthcaldt"
    RETURN_COLUMN = "mret"
elif (
    "msf" in available
    and {"permno", "date", "ret"}.issubset(available["msf"])
):
    RETURN_TABLE = "crsp.msf"
    DATE_COLUMN = "date"
    RETURN_COLUMN = "ret"
else:
    raise RuntimeError(
        "Neither supported CRSP monthly layout was found. "
        "Inspect crsp_schema_discovery.csv."
    )

print()
print("Selected return source:", RETURN_TABLE)
print("Date field:", DATE_COLUMN)
print("Total-return field:", RETURN_COLUMN)
print()

# ------------------------------------------------------------
# 4. Find every CRSP security record for the requested tickers
# ------------------------------------------------------------

ticker_sql = ", ".join(f"'{ticker}'" for ticker in ALL_TICKERS)

names_sql = f"""
SELECT
    permno,
    upper(ticker) AS ticker,
    comnam,
    namedt,
    nameenddt,
    shrcd,
    exchcd
FROM crsp.stocknames
WHERE upper(ticker) IN ({ticker_sql})
ORDER BY ticker, permno, namedt
"""

names = db.raw_sql(names_sql)

if names.empty:
    raise RuntimeError("No requested tickers were found in crsp.stocknames.")

names["permno"] = pd.to_numeric(names["permno"], errors="coerce").astype("Int64")
names["namedt"] = pd.to_datetime(names["namedt"], errors="coerce")
names["nameenddt"] = pd.to_datetime(names["nameenddt"], errors="coerce")

names.to_csv(OUTDIR / "instrument_name_history_all_candidates.csv", index=False)

print("=" * 100)
print("A. ALL CRSP NAME-HISTORY MATCHES")
print("=" * 100)
display(names)

found_tickers = set(names["ticker"].dropna())
missing_tickers = sorted(set(ALL_TICKERS) - found_tickers)

if missing_tickers:
    print("WARNING — TICKERS NOT FOUND:", missing_tickers)

permnos = sorted(
    names["permno"].dropna().astype(int).unique().tolist()
)

if not permnos:
    raise RuntimeError("No valid PERMNO values were recovered.")

# ------------------------------------------------------------
# 5. Pull monthly CRSP total returns for every candidate PERMNO
# ------------------------------------------------------------

permno_sql = ", ".join(str(x) for x in permnos)

returns_sql = f"""
SELECT
    permno,
    {DATE_COLUMN} AS observation_date,
    {RETURN_COLUMN} AS total_return
FROM {RETURN_TABLE}
WHERE permno IN ({permno_sql})
ORDER BY permno, {DATE_COLUMN}
"""

returns_raw = db.raw_sql(returns_sql)

returns_raw["permno"] = pd.to_numeric(
    returns_raw["permno"], errors="coerce"
).astype("Int64")

returns_raw["observation_date"] = pd.to_datetime(
    returns_raw["observation_date"], errors="coerce"
)

returns_raw["total_return"] = pd.to_numeric(
    returns_raw["total_return"], errors="coerce"
)

returns_raw = returns_raw.dropna(
    subset=["permno", "observation_date"]
).copy()

# Standardize every observation to its calendar month-end.
returns_raw["month"] = (
    returns_raw["observation_date"]
    .dt.to_period("M")
    .dt.to_timestamp("M")
)

returns_raw.to_parquet(
    OUTDIR / "crsp_monthly_returns_all_candidate_permnos.parquet",
    index=False
)

print()
print("Monthly rows pulled:", len(returns_raw))
print(
    "Raw return date range:",
    returns_raw["month"].min(),
    "to",
    returns_raw["month"].max()
)

# ------------------------------------------------------------
# 6. Check duplicate PERMNO-month observations
# ------------------------------------------------------------

duplicates = (
    returns_raw.groupby(["permno", "month"])
    .size()
    .rename("row_count")
    .reset_index()
    .query("row_count > 1")
)

duplicates.to_csv(
    OUTDIR / "duplicate_permno_months.csv",
    index=False
)

if duplicates.empty:
    print("Duplicate PERMNO-month check: PASS")
else:
    print("Duplicate PERMNO-month check: FAIL")
    display(duplicates)

# Do not silently average duplicate returns.
if not duplicates.empty:
    raise RuntimeError(
        "Duplicate PERMNO-month rows exist. "
        "They must be resolved before continuing."
    )

# ------------------------------------------------------------
# 7. Measure usable history for every ticker-PERMNO candidate
# ------------------------------------------------------------

candidate_rows = []

for (ticker, permno), grp in names.groupby(["ticker", "permno"]):
    permno = int(permno)

    name_start = grp["namedt"].min()
    name_end = grp["nameenddt"].max()

    r = returns_raw.loc[
        returns_raw["permno"].astype(int).eq(permno)
    ].copy()

    if pd.notna(name_start):
        r = r.loc[r["month"] >= name_start.to_period("M").to_timestamp("M")]

    if pd.notna(name_end):
        r = r.loc[r["month"] <= name_end.to_period("M").to_timestamp("M")]

    valid = r.dropna(subset=["total_return"])

    candidate_rows.append(
        {
            "ticker": ticker,
            "permno": permno,
            "company_name": " | ".join(
                sorted(grp["comnam"].dropna().astype(str).unique())
            ),
            "name_history_start": name_start,
            "name_history_end": name_end,
            "first_valid_return": (
                valid["month"].min() if not valid.empty else pd.NaT
            ),
            "last_valid_return": (
                valid["month"].max() if not valid.empty else pd.NaT
            ),
            "valid_months": len(valid),
            "missing_return_rows": int(r["total_return"].isna().sum()),
            "name_history_records": len(grp),
        }
    )

candidate_summary = pd.DataFrame(candidate_rows).sort_values(
    ["ticker", "valid_months", "last_valid_return"],
    ascending=[True, False, False]
)

candidate_summary.to_csv(
    OUTDIR / "ticker_permno_candidate_summary.csv",
    index=False
)

print()
print("=" * 100)
print("B. TICKER–PERMNO CANDIDATE SUMMARY")
print("=" * 100)
display(candidate_summary)

# ------------------------------------------------------------
# 8. Provisional selection: longest valid return history
#
# This is explicitly provisional. The report preserves every
# candidate so no identifier ambiguity is hidden.
# ------------------------------------------------------------

selected = (
    candidate_summary
    .sort_values(
        ["ticker", "valid_months", "last_valid_return"],
        ascending=[True, False, False]
    )
    .groupby("ticker", as_index=False)
    .first()
)

candidate_counts = (
    candidate_summary.groupby("ticker")["permno"]
    .nunique()
    .rename("candidate_permno_count")
    .reset_index()
)

selected = selected.merge(candidate_counts, on="ticker", how="left")
selected["identifier_status"] = np.where(
    selected["candidate_permno_count"].eq(1),
    "UNAMBIGUOUS",
    "REVIEW_MULTIPLE_PERMNOS"
)

selected.to_csv(
    OUTDIR / "provisional_instrument_map.csv",
    index=False
)

print()
print("=" * 100)
print("C. PROVISIONAL INSTRUMENT MAP")
print("=" * 100)
display(
    selected[
        [
            "ticker",
            "permno",
            "company_name",
            "first_valid_return",
            "last_valid_return",
            "valid_months",
            "candidate_permno_count",
            "identifier_status",
        ]
    ]
)

ambiguous = selected.loc[
    selected["identifier_status"] != "UNAMBIGUOUS"
]

if not ambiguous.empty:
    print()
    print(
        "ATTENTION: At least one ticker has multiple PERMNO candidates. "
        "Do not freeze identifiers until these rows are reviewed."
    )

# ------------------------------------------------------------
# 9. Build the no-fill diagnostic return panel
# ------------------------------------------------------------

series_list = []
selection_lookup = selected.set_index("ticker")

for ticker in ALL_TICKERS:
    if ticker not in selection_lookup.index:
        continue

    info = selection_lookup.loc[ticker]
    permno = int(info["permno"])

    r = returns_raw.loc[
        returns_raw["permno"].astype(int).eq(permno),
        ["month", "total_return"]
    ].copy()

    start = info["name_history_start"]
    end = info["name_history_end"]

    if pd.notna(start):
        r = r.loc[r["month"] >= start.to_period("M").to_timestamp("M")]

    if pd.notna(end):
        r = r.loc[r["month"] <= end.to_period("M").to_timestamp("M")]

    r = r.drop_duplicates("month").set_index("month")
    r = r.rename(columns={"total_return": ticker})
    series_list.append(r[[ticker]])

panel = pd.concat(series_list, axis=1).sort_index()
panel.index.name = "month"

# IMPORTANT:
# No backward fill.
# No forward fill.
# No replacement of missing returns with zero.
panel.to_parquet(
    OUTDIR / "diagnostic_monthly_total_return_panel_NO_FILL.parquet"
)

# ------------------------------------------------------------
# 10. Return-quality and missing-month audit
# ------------------------------------------------------------

quality_rows = []
gap_rows = []
extreme_rows = []

for ticker in panel.columns:
    s = panel[ticker].dropna()

    if s.empty:
        continue

    expected_months = pd.date_range(
        s.index.min(),
        s.index.max(),
        freq="ME"
    )

    missing_internal = expected_months.difference(s.index)

    quality_rows.append(
        {
            "ticker": ticker,
            "first_valid_month": s.index.min(),
            "last_valid_month": s.index.max(),
            "valid_months": int(s.notna().sum()),
            "internal_missing_months": len(missing_internal),
            "zero_returns": int(s.eq(0).sum()),
            "minimum_return": float(s.min()),
            "maximum_return": float(s.max()),
            "mean_monthly_return": float(s.mean()),
            "monthly_volatility": float(s.std(ddof=1)),
            "returns_le_minus_100pct": int(s.le(-1.0).sum()),
            "absolute_returns_gt_50pct": int(s.abs().gt(0.50).sum()),
        }
    )

    for month in missing_internal:
        gap_rows.append(
            {"ticker": ticker, "missing_month": month}
        )

    extreme = s.loc[s.abs() > 0.50]
    for month, value in extreme.items():
        extreme_rows.append(
            {
                "ticker": ticker,
                "month": month,
                "total_return": value,
                "reason": "absolute monthly return > 50%",
            }
        )

quality = pd.DataFrame(quality_rows)
gaps = pd.DataFrame(
    gap_rows,
    columns=["ticker", "missing_month"]
)
extremes = pd.DataFrame(
    extreme_rows,
    columns=["ticker", "month", "total_return", "reason"]
)

quality.to_csv(OUTDIR / "return_quality_summary.csv", index=False)
gaps.to_csv(OUTDIR / "internal_missing_months.csv", index=False)
extremes.to_csv(OUTDIR / "extreme_return_observations.csv", index=False)

print()
print("=" * 100)
print("D. RETURN QUALITY")
print("=" * 100)
display(quality)

if not gaps.empty:
    print("Internal missing months:")
    display(gaps)
else:
    print("Internal missing-month check: PASS")

if not extremes.empty:
    print()
    print("Extreme observations requiring source verification:")
    display(extremes)
else:
    print("Absolute monthly return > 50% check: PASS")

# ------------------------------------------------------------
# 11. Calculate common samples and model feasibility
# ------------------------------------------------------------

def sample_feasibility(label, asset_tickers):
    required = asset_tickers + BENCHMARKS
    missing_columns = [c for c in required if c not in panel.columns]

    if missing_columns:
        return {
            "design": label,
            "assets": ", ".join(asset_tickers),
            "required_columns_missing": ", ".join(missing_columns),
            "common_first_month": pd.NaT,
            "common_last_month": pd.NaT,
            "common_months": 0,
            "internal_common_sample_gaps": np.nan,
            "first_state_after_12m_lookback": pd.NaT,
            "last_state_with_next_month_target": pd.NaT,
            "model_rows": 0,
            "training_rows_through_2018": 0,
            "oos_rows_from_2019": 0,
        }

    common = panel[required].dropna(how="any").copy()

    if common.empty:
        return {
            "design": label,
            "assets": ", ".join(asset_tickers),
            "required_columns_missing": "",
            "common_first_month": pd.NaT,
            "common_last_month": pd.NaT,
            "common_months": 0,
            "internal_common_sample_gaps": np.nan,
            "first_state_after_12m_lookback": pd.NaT,
            "last_state_with_next_month_target": pd.NaT,
            "model_rows": 0,
            "training_rows_through_2018": 0,
            "oos_rows_from_2019": 0,
        }

    expected = pd.date_range(
        common.index.min(),
        common.index.max(),
        freq="ME"
    )
    internal_gaps = len(expected.difference(common.index))

    # A state needs 12 prior months and a next-month target.
    if len(common) > LOOKBACK_MONTHS + 1:
        model_index = common.index[LOOKBACK_MONTHS:-1]
    else:
        model_index = pd.DatetimeIndex([])

    train_rows = int((model_index <= TRAIN_END).sum())
    oos_rows = int((model_index >= OOS_START).sum())

    return {
        "design": label,
        "assets": ", ".join(asset_tickers),
        "required_columns_missing": "",
        "common_first_month": common.index.min(),
        "common_last_month": common.index.max(),
        "common_months": len(common),
        "internal_common_sample_gaps": internal_gaps,
        "first_state_after_12m_lookback": (
            model_index.min() if len(model_index) else pd.NaT
        ),
        "last_state_with_next_month_target": (
            model_index.max() if len(model_index) else pd.NaT
        ),
        "model_rows": len(model_index),
        "training_rows_through_2018": train_rows,
        "oos_rows_from_2019": oos_rows,
    }

feasibility = pd.DataFrame(
    [
        sample_feasibility(
            "Primary candidate: base metals",
            ASSETS_BASE_METALS
        ),
        sample_feasibility(
            "Alternative candidate: copper",
            ASSETS_COPPER
        ),
    ]
)

feasibility.to_csv(
    OUTDIR / "sample_feasibility_comparison.csv",
    index=False
)

print()
print("=" * 100)
print("E. SAMPLE FEASIBILITY")
print("=" * 100)
display(feasibility)

# ------------------------------------------------------------
# 12. Reviewer gate summary
# ------------------------------------------------------------

duplicate_pass = duplicates.empty
impossible_pass = (
    quality["returns_le_minus_100pct"].sum() == 0
    if not quality.empty else False
)
internal_gap_pass = (
    quality["internal_missing_months"].sum() == 0
    if not quality.empty else False
)
all_tickers_pass = len(missing_tickers) == 0
identifier_pass = (
    selected["identifier_status"].eq("UNAMBIGUOUS").all()
    if not selected.empty else False
)

gate = pd.DataFrame(
    [
        {
            "gate": "All candidate tickers located",
            "status": "PASS" if all_tickers_pass else "FAIL",
            "detail": (
                "All found"
                if all_tickers_pass
                else f"Missing: {missing_tickers}"
            ),
        },
        {
            "gate": "Ticker-to-PERMNO mapping unambiguous",
            "status": "PASS" if identifier_pass else "REVIEW",
            "detail": "See provisional_instrument_map.csv",
        },
        {
            "gate": "No duplicate PERMNO-month rows",
            "status": "PASS" if duplicate_pass else "FAIL",
            "detail": f"{len(duplicates)} duplicate keys",
        },
        {
            "gate": "No impossible returns <= -100%",
            "status": "PASS" if impossible_pass else "FAIL",
            "detail": (
                f"{int(quality['returns_le_minus_100pct'].sum())} observations"
                if not quality.empty else "No data"
            ),
        },
        {
            "gate": "No unexplained internal missing months",
            "status": "PASS" if internal_gap_pass else "REVIEW",
            "detail": (
                f"{int(quality['internal_missing_months'].sum())} gaps"
                if not quality.empty else "No data"
            ),
        },
        {
            "gate": "No backfill or zero imputation",
            "status": "PASS",
            "detail": "Diagnostic panel constructed without any fill operation",
        },
        {
            "gate": "Macro publication-lag audit",
            "status": "NOT RUN",
            "detail": "Requires the next audit cell",
        },
        {
            "gate": "Feature/target leakage audit",
            "status": "NOT RUN",
            "detail": "Requires the frozen feature builder",
        },
        {
            "gate": "Transaction-cost and turnover audit",
            "status": "NOT RUN",
            "detail": "Requires strategy weights and returns",
        },
    ]
)

gate.to_csv(OUTDIR / "reviewer_gate_summary.csv", index=False)

print()
print("=" * 100)
print("F. REVIEWER GATE SUMMARY")
print("=" * 100)
display(gate)

# ------------------------------------------------------------
# 13. Create hashes for every generated audit artifact
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

hash_rows = []

for path in sorted(OUTDIR.iterdir()):
    if path.is_file():
        hash_rows.append(
            {
                "filename": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

hash_manifest = pd.DataFrame(hash_rows)
hash_manifest.to_csv(
    OUTDIR / "audit_file_hashes.csv",
    index=False
)

run_metadata = {
    "run_id": RUN_ID,
    "python_version": sys.version,
    "pandas_version": pd.__version__,
    "wrds_return_table": RETURN_TABLE,
    "wrds_date_column": DATE_COLUMN,
    "wrds_return_column": RETURN_COLUMN,
    "tickers_requested": ALL_TICKERS,
    "train_end": str(TRAIN_END.date()),
    "oos_start": str(OOS_START.date()),
    "lookback_months": LOOKBACK_MONTHS,
    "missing_data_rule": "No backfill, no forward fill, no zero imputation",
    "return_interpretation": "CRSP monthly security total return",
}

with open(OUTDIR / "run_metadata.json", "w") as stream:
    json.dump(run_metadata, stream, indent=2)

db.close()

print()
print("=" * 100)
print("AUDIT COMPLETE")
print("=" * 100)
print("Saved to:", OUTDIR)
print()
print("COPY BACK THESE FOUR OUTPUT SECTIONS:")
print("B. TICKER–PERMNO CANDIDATE SUMMARY")
print("C. PROVISIONAL INSTRUMENT MAP")
print("D. RETURN QUALITY")
print("E. SAMPLE FEASIBILITY")
print()
print("Do not run the old PPO notebook yet.")

In [ ]:
# ============================================================
# REVIEWER DATA AUDIT 2
# Correct identifiers + legacy/CIZ comparison + final sample
# ============================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import wrds
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

OUTDIR = Path.home() / "gsci_reviewer_audit_20260722_205114"

CORRECT_PERMNO = {
    "CPER": 13102,
    "DBA": 91712,
    "DBB": 91715,
    "DBC": 91129,
    "DBE": 91709,
    "GLD": 90448,
    "GSG": 91381,
}

EXPECTED_NAME = {
    "CPER": "UNITED STATES",
    "DBA": "DB MULTI SECT",
    "DBB": "DB MULTI SECT",
    "DBC": "COMMODITY INDEX",
    "DBE": "DB MULTI SECT",
    "GLD": "GOLD TRUST",
    "GSG": "GSCI",
}

BASE_METALS_ASSETS = ["DBE", "GLD", "DBA", "DBB"]
COPPER_ASSETS = ["DBE", "GLD", "DBA", "CPER"]
BENCHMARKS = ["GSG", "DBC"]

TRAIN_END = pd.Timestamp("2018-12-31")
OOS_START = pd.Timestamp("2019-01-31")
LOOKBACK = 12

print("=" * 100)
print("CORRECTED IDENTIFIER AND RETURN-SOURCE AUDIT")
print("=" * 100)

db = wrds.Connection()

# ------------------------------------------------------------
# 1. Inspect exact columns in both monthly CRSP tables
# ------------------------------------------------------------

schema = db.raw_sql("""
    SELECT
        lower(table_name) AS table_name,
        lower(column_name) AS column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'crsp'
      AND lower(table_name) IN ('msf', 'msf_v2')
    ORDER BY table_name, ordinal_position
""")

schema.to_csv(OUTDIR / "monthly_table_column_audit.csv", index=False)

print("\nA. MONTHLY TABLE COLUMNS")
for table in ["msf", "msf_v2"]:
    cols = schema.loc[
        schema["table_name"].eq(table), "column_name"
    ].tolist()
    print(f"\n{table}:")
    print(cols)

table_columns = (
    schema.groupby("table_name")["column_name"]
    .apply(set)
    .to_dict()
)

# Recognize both legacy and CIZ field names.
if (
    "msf_v2" in table_columns
    and "permno" in table_columns["msf_v2"]
    and "mthcaldt" in table_columns["msf_v2"]
    and "mthret" in table_columns["msf_v2"]
):
    CIZ_AVAILABLE = True
    CIZ_DATE = "mthcaldt"
    CIZ_RETURN = "mthret"
elif (
    "msf_v2" in table_columns
    and "permno" in table_columns["msf_v2"]
    and "mthcaldt" in table_columns["msf_v2"]
    and "mret" in table_columns["msf_v2"]
):
    CIZ_AVAILABLE = True
    CIZ_DATE = "mthcaldt"
    CIZ_RETURN = "mret"
else:
    CIZ_AVAILABLE = False
    CIZ_DATE = None
    CIZ_RETURN = None

LEGACY_AVAILABLE = (
    "msf" in table_columns
    and {"permno", "date", "ret"}.issubset(table_columns["msf"])
)

print("\nCIZ msf_v2 usable:", CIZ_AVAILABLE)
print("Legacy msf usable:", LEGACY_AVAILABLE)

if CIZ_AVAILABLE:
    print("CIZ date field:", CIZ_DATE)
    print("CIZ return field:", CIZ_RETURN)

if not CIZ_AVAILABLE and not LEGACY_AVAILABLE:
    raise RuntimeError("No supported monthly return table was found.")

# ------------------------------------------------------------
# 2. Verify exact selected securities against name history
# ------------------------------------------------------------

permno_list = ", ".join(str(x) for x in CORRECT_PERMNO.values())

selected_names = db.raw_sql(f"""
    SELECT
        permno,
        upper(ticker) AS ticker,
        comnam,
        namedt,
        nameenddt,
        shrcd,
        exchcd
    FROM crsp.stocknames
    WHERE permno IN ({permno_list})
    ORDER BY permno, namedt
""")

selected_names["permno"] = pd.to_numeric(
    selected_names["permno"], errors="coerce"
).astype("Int64")
selected_names["namedt"] = pd.to_datetime(
    selected_names["namedt"], errors="coerce"
)
selected_names["nameenddt"] = pd.to_datetime(
    selected_names["nameenddt"], errors="coerce"
)

selected_names.to_csv(
    OUTDIR / "verified_selected_security_name_history.csv",
    index=False
)

identity_rows = []

for ticker, permno in CORRECT_PERMNO.items():
    rows = selected_names.loc[
        selected_names["permno"].astype(int).eq(permno)
        & selected_names["ticker"].eq(ticker)
    ].copy()

    names_text = " | ".join(
        sorted(rows["comnam"].dropna().astype(str).unique())
    )

    expected_match = EXPECTED_NAME[ticker] in names_text.upper()

    identity_rows.append(
        {
            "ticker": ticker,
            "permno": permno,
            "company_names": names_text,
            "first_name_date": (
                rows["namedt"].min() if not rows.empty else pd.NaT
            ),
            "last_name_date": (
                rows["nameenddt"].max() if not rows.empty else pd.NaT
            ),
            "share_codes": ", ".join(
                sorted(rows["shrcd"].dropna().astype(str).unique())
            ),
            "exchange_codes": ", ".join(
                sorted(rows["exchcd"].dropna().astype(str).unique())
            ),
            "expected_identity_text_found": expected_match,
            "identity_status": (
                "PASS" if expected_match else "REVIEW"
            ),
        }
    )

identity = pd.DataFrame(identity_rows).sort_values("ticker")
identity.to_csv(
    OUTDIR / "frozen_instrument_map.csv",
    index=False
)

print("\n" + "=" * 100)
print("B. CORRECTED FROZEN INSTRUMENT MAP")
print("=" * 100)
display(identity)

if not identity["identity_status"].eq("PASS").all():
    print("WARNING: At least one identity requires manual review.")

# ------------------------------------------------------------
# 3. Pull a monthly table using exact PERMNOs
# ------------------------------------------------------------

def pull_returns(table, date_column, return_column, source_name):
    query = f"""
        SELECT
            permno,
            {date_column} AS observation_date,
            {return_column} AS total_return
        FROM {table}
        WHERE permno IN ({permno_list})
        ORDER BY permno, {date_column}
    """

    result = db.raw_sql(query)

    result["permno"] = pd.to_numeric(
        result["permno"], errors="coerce"
    ).astype("Int64")

    result["observation_date"] = pd.to_datetime(
        result["observation_date"], errors="coerce"
    )

    result["total_return"] = pd.to_numeric(
        result["total_return"], errors="coerce"
    )

    result = result.dropna(
        subset=["permno", "observation_date"]
    ).copy()

    result["month"] = (
        result["observation_date"]
        .dt.to_period("M")
        .dt.to_timestamp("M")
    )

    result["source"] = source_name
    return result

legacy = pd.DataFrame()
ciz = pd.DataFrame()

if LEGACY_AVAILABLE:
    legacy = pull_returns(
        "crsp.msf",
        "date",
        "ret",
        "legacy_msf_ret"
    )

if CIZ_AVAILABLE:
    ciz = pull_returns(
        "crsp.msf_v2",
        CIZ_DATE,
        CIZ_RETURN,
        f"ciz_msf_v2_{CIZ_RETURN}"
    )

# ------------------------------------------------------------
# 4. Compare source coverage
# ------------------------------------------------------------

coverage_rows = []

for source_name, frame in [
    ("legacy_msf_ret", legacy),
    (
        f"ciz_msf_v2_{CIZ_RETURN}"
        if CIZ_AVAILABLE else "ciz_unavailable",
        ciz
    ),
]:
    if frame.empty:
        continue

    for ticker, permno in CORRECT_PERMNO.items():
        s = frame.loc[
            frame["permno"].astype(int).eq(permno)
        ].copy()

        valid = s.dropna(subset=["total_return"])

        coverage_rows.append(
            {
                "source": source_name,
                "ticker": ticker,
                "permno": permno,
                "first_row": s["month"].min(),
                "last_row": s["month"].max(),
                "first_valid_return": valid["month"].min(),
                "last_valid_return": valid["month"].max(),
                "rows": len(s),
                "valid_returns": len(valid),
                "missing_returns": int(
                    s["total_return"].isna().sum()
                ),
                "duplicate_months": int(
                    s.duplicated(["permno", "month"]).sum()
                ),
            }
        )

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(
    OUTDIR / "legacy_ciz_coverage_comparison.csv",
    index=False
)

print("\n" + "=" * 100)
print("C. LEGACY/CIZ COVERAGE")
print("=" * 100)
display(coverage)

# ------------------------------------------------------------
# 5. Compare overlapping legacy and CIZ returns
# ------------------------------------------------------------

if not legacy.empty and not ciz.empty:
    comparison = legacy[
        ["permno", "month", "total_return"]
    ].merge(
        ciz[["permno", "month", "total_return"]],
        on=["permno", "month"],
        how="inner",
        suffixes=("_legacy", "_ciz")
    )

    comparison["absolute_difference"] = (
        comparison["total_return_legacy"]
        - comparison["total_return_ciz"]
    ).abs()

    comparison["ticker"] = comparison["permno"].map(
        {v: k for k, v in CORRECT_PERMNO.items()}
    )

    comparison_summary = (
        comparison.groupby(["ticker", "permno"])
        .agg(
            overlap_months=("month", "size"),
            mean_absolute_difference=(
                "absolute_difference", "mean"
            ),
            maximum_absolute_difference=(
                "absolute_difference", "max"
            ),
            differences_gt_1e_8=(
                "absolute_difference",
                lambda x: int((x > 1e-8).sum())
            ),
        )
        .reset_index()
    )

    comparison.to_parquet(
        OUTDIR / "legacy_ciz_return_comparison.parquet",
        index=False
    )
    comparison_summary.to_csv(
        OUTDIR / "legacy_ciz_return_comparison_summary.csv",
        index=False
    )

    print("\n" + "=" * 100)
    print("D. LEGACY/CIZ RETURN DIFFERENCES")
    print("=" * 100)
    display(comparison_summary)
else:
    comparison_summary = pd.DataFrame()
    print("\nD. Legacy/CIZ comparison unavailable.")

# Prefer CIZ when available because it is the newer CRSP format.
if not ciz.empty:
    chosen = ciz.copy()
    CHOSEN_SOURCE = f"crsp.msf_v2.{CIZ_RETURN}"
else:
    chosen = legacy.copy()
    CHOSEN_SOURCE = "crsp.msf.ret"

print("\nChosen final source:", CHOSEN_SOURCE)

# ------------------------------------------------------------
# 6. Build correct ticker panel
# ------------------------------------------------------------

series = []

for ticker, permno in CORRECT_PERMNO.items():
    s = chosen.loc[
        chosen["permno"].astype(int).eq(permno),
        ["month", "total_return"]
    ].copy()

    duplicate_count = s.duplicated("month").sum()
    if duplicate_count:
        raise RuntimeError(
            f"{ticker}: {duplicate_count} duplicate months found."
        )

    s = (
        s.dropna(subset=["total_return"])
        .set_index("month")
        .rename(columns={"total_return": ticker})
    )

    series.append(s[[ticker]])

panel = pd.concat(series, axis=1).sort_index()
panel.index.name = "month"

# No imputation of any kind.
panel.to_parquet(
    OUTDIR / "verified_monthly_total_return_panel_NO_FILL.parquet"
)

# ------------------------------------------------------------
# 7. Corrected quality report
# ------------------------------------------------------------

quality_rows = []

for ticker in panel.columns:
    s = panel[ticker].dropna()

    expected = pd.date_range(
        s.index.min(),
        s.index.max(),
        freq="ME"
    )

    quality_rows.append(
        {
            "ticker": ticker,
            "permno": CORRECT_PERMNO[ticker],
            "first_valid_month": s.index.min(),
            "last_valid_month": s.index.max(),
            "valid_months": len(s),
            "internal_missing_months": len(
                expected.difference(s.index)
            ),
            "zero_returns": int(s.eq(0).sum()),
            "minimum_return": float(s.min()),
            "maximum_return": float(s.max()),
            "returns_le_minus_100pct": int(
                s.le(-1.0).sum()
            ),
            "absolute_returns_gt_50pct": int(
                s.abs().gt(0.50).sum()
            ),
        }
    )

corrected_quality = pd.DataFrame(quality_rows)
corrected_quality.to_csv(
    OUTDIR / "corrected_return_quality_summary.csv",
    index=False
)

print("\n" + "=" * 100)
print("E. CORRECTED RETURN QUALITY")
print("=" * 100)
display(corrected_quality)

# ------------------------------------------------------------
# 8. Correct sample feasibility
# ------------------------------------------------------------

def feasibility(label, assets):
    required = assets + BENCHMARKS
    common = panel[required].dropna(how="any")

    expected = pd.date_range(
        common.index.min(),
        common.index.max(),
        freq="ME"
    )

    common_gaps = len(expected.difference(common.index))

    if len(common) > LOOKBACK + 1:
        state_index = common.index[LOOKBACK:-1]
    else:
        state_index = pd.DatetimeIndex([])

    return {
        "design": label,
        "assets": ", ".join(assets),
        "benchmarks": ", ".join(BENCHMARKS),
        "common_first_month": common.index.min(),
        "common_last_month": common.index.max(),
        "common_months": len(common),
        "common_sample_gaps": common_gaps,
        "first_usable_state": (
            state_index.min() if len(state_index) else pd.NaT
        ),
        "last_usable_state": (
            state_index.max() if len(state_index) else pd.NaT
        ),
        "model_rows": len(state_index),
        "training_rows_through_2018": int(
            (state_index <= TRAIN_END).sum()
        ),
        "oos_rows_from_2019": int(
            (state_index >= OOS_START).sum()
        ),
        "features_if_64": 64,
        "training_rows_per_64_features": (
            float((state_index <= TRAIN_END).sum() / 64)
            if len(state_index) else np.nan
        ),
    }

sample_comparison = pd.DataFrame(
    [
        feasibility(
            "DBB base-metals primary candidate",
            BASE_METALS_ASSETS
        ),
        feasibility(
            "CPER copper alternative",
            COPPER_ASSETS
        ),
    ]
)

sample_comparison.to_csv(
    OUTDIR / "corrected_sample_feasibility.csv",
    index=False
)

print("\n" + "=" * 100)
print("F. CORRECTED SAMPLE FEASIBILITY")
print("=" * 100)
display(sample_comparison)

# ------------------------------------------------------------
# 9. Preliminary decision gate
# ------------------------------------------------------------

base = sample_comparison.iloc[0]
copper = sample_comparison.iloc[1]

decision = pd.DataFrame(
    [
        {
            "question": "Can the 2010–April 2026 paper be reproduced from this WRDS source?",
            "answer": (
                "NO"
                if panel.index.max() < pd.Timestamp("2026-04-30")
                else "POSSIBLY"
            ),
            "evidence": f"Latest verified return month: {panel.index.max().date()}",
        },
        {
            "question": "Does CPER provide the same long training history as DBB?",
            "answer": (
                "NO"
                if copper["training_rows_through_2018"]
                < base["training_rows_through_2018"]
                else "YES"
            ),
            "evidence": (
                f"DBB design: {int(base['training_rows_through_2018'])} "
                f"training rows; CPER design: "
                f"{int(copper['training_rows_through_2018'])}"
            ),
        },
        {
            "question": "Is a 64-feature deep-RL claim well supported by the sample?",
            "answer": "NO — SAMPLE IS SMALL",
            "evidence": (
                f"DBB training rows/features ratio: "
                f"{base['training_rows_per_64_features']:.2f}; "
                f"CPER ratio: "
                f"{copper['training_rows_per_64_features']:.2f}"
            ),
        },
        {
            "question": "Recommended primary investable universe",
            "answer": "DBE, GLD, DBA, DBB",
            "evidence": (
                "Longer common history; CPER retained as a shorter-sample robustness test"
            ),
        },
    ]
)

decision.to_csv(
    OUTDIR / "preliminary_scientific_decision.csv",
    index=False
)

print("\n" + "=" * 100)
print("G. PRELIMINARY SCIENTIFIC DECISION")
print("=" * 100)
display(decision)

# ------------------------------------------------------------
# 10. Update hashes
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

hash_rows = []

for path in sorted(OUTDIR.iterdir()):
    if path.is_file() and path.name != "audit_file_hashes_updated.csv":
        hash_rows.append(
            {
                "filename": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

pd.DataFrame(hash_rows).to_csv(
    OUTDIR / "audit_file_hashes_updated.csv",
    index=False
)

db.close()

print("\n" + "=" * 100)
print("CORRECTED AUDIT COMPLETE")
print("=" * 100)
print("Final return source:", CHOSEN_SOURCE)
print("Saved to:", OUTDIR)
print()
print("PASTE BACK SECTIONS:")
print("A. MONTHLY TABLE COLUMNS")
print("C. LEGACY/CIZ COVERAGE")
print("D. LEGACY/CIZ RETURN DIFFERENCES")
print("F. CORRECTED SAMPLE FEASIBILITY")
print("G. PRELIMINARY SCIENTIFIC DECISION")

In [ ]:
# ============================================================
# REVIEWER DATA AUDIT 3
# Frozen features, macro timing, target alignment, leakage tests
# ============================================================

from pathlib import Path
from io import StringIO
import hashlib
import json
import urllib.request

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 200)

OUTDIR = Path.home() / "gsci_reviewer_audit_20260722_205114"

PANEL_PATH = (
    OUTDIR / "verified_monthly_total_return_panel_NO_FILL.parquet"
)

ASSETS = ["DBE", "GLD", "DBA", "DBB"]
BENCHMARKS = ["GSG", "DBC"]

TRAIN_TARGET_END = pd.Timestamp("2018-12-31")
OOS_TARGET_START = pd.Timestamp("2019-01-31")
FINAL_TARGET_END = pd.Timestamp("2025-12-31")

print("=" * 100)
print("FEATURE, MACRO-TIMING, AND LEAKAGE AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# 1. Load and validate the frozen CRSP panel
# ------------------------------------------------------------

if not PANEL_PATH.exists():
    raise FileNotFoundError(PANEL_PATH)

returns = pd.read_parquet(PANEL_PATH).sort_index()
returns.index = pd.to_datetime(returns.index)
returns.index.name = "return_month"

required_columns = ASSETS + BENCHMARKS
missing_columns = [
    col for col in required_columns if col not in returns.columns
]

if missing_columns:
    raise RuntimeError(f"Missing return columns: {missing_columns}")

returns = returns[required_columns].copy()

if returns.index.duplicated().any():
    raise RuntimeError("Duplicate monthly dates exist.")

expected_months = pd.date_range(
    returns.index.min(),
    returns.index.max(),
    freq="ME"
)

missing_calendar_months = expected_months.difference(returns.index)

if len(missing_calendar_months):
    raise RuntimeError(
        f"Missing calendar months: {missing_calendar_months.tolist()}"
    )

if returns[required_columns].isna().any().any():
    print("Raw panel contains missing values by ticker:")
    display(returns.isna().sum().to_frame("missing_values"))
else:
    print("Frozen return panel completeness: PASS")

print(
    "Frozen return period:",
    returns.index.min().date(),
    "to",
    returns.index.max().date()
)

# ------------------------------------------------------------
# 2. Compact primary market feature set
#
# Three features per asset:
#   6-month compounded momentum
#   12-month compounded momentum
#   12-month realized volatility
#
# Total primary features: 4 × 3 = 12
# ------------------------------------------------------------

def build_market_features(asset_returns):
    features = pd.DataFrame(index=asset_returns.index)

    for ticker in ASSETS:
        r = asset_returns[ticker]

        features[f"state__{ticker.lower()}__momentum_6m"] = (
            (1.0 + r).rolling(
                window=6,
                min_periods=6
            ).apply(np.prod, raw=True) - 1.0
        )

        features[f"state__{ticker.lower()}__momentum_12m"] = (
            (1.0 + r).rolling(
                window=12,
                min_periods=12
            ).apply(np.prod, raw=True) - 1.0
        )

        features[f"state__{ticker.lower()}__volatility_12m"] = (
            r.rolling(
                window=12,
                min_periods=12
            ).std(ddof=1)
        )

    return features

market_features = build_market_features(returns[ASSETS])

if market_features.shape[1] != 12:
    raise RuntimeError(
        f"Expected 12 market features; found {market_features.shape[1]}"
    )

market_features.to_parquet(
    OUTDIR / "primary_market_features_12.parquet"
)

print("\nPrimary market feature count:", market_features.shape[1])
print("Primary feature names:")
for name in market_features.columns:
    print(" ", name)

# ------------------------------------------------------------
# 3. Obtain public FRED series without embedding an API key
#
# These macro variables are SECONDARY sensitivity features.
# The confirmatory primary specification remains market-only,
# avoiding dependence on ex-post-revised macro observations.
# ------------------------------------------------------------

FRED_SPEC = {
    "CPIAUCSL": {
        "description": "Consumer Price Index, all urban consumers",
        "frequency": "monthly",
        "transform": "12-month percentage change",
        "publication_lag_months": 1,
        "feature": "state__macro__cpi_inflation_yoy_lag1",
        "primary_or_secondary": "secondary sensitivity only",
        "revision_warning": (
            "Current FRED vintage may contain historical revisions"
        ),
    },
    "INDPRO": {
        "description": "Industrial Production Index",
        "frequency": "monthly",
        "transform": "12-month percentage change",
        "publication_lag_months": 1,
        "feature": "state__macro__industrial_production_yoy_lag1",
        "primary_or_secondary": "secondary sensitivity only",
        "revision_warning": (
            "Current FRED vintage may contain historical revisions"
        ),
    },
    "DTWEXBGS": {
        "description": "Trade Weighted U.S. Dollar Index: Broad",
        "frequency": "daily",
        "transform": "month-end level, then 12-month percentage change",
        "publication_lag_months": 1,
        "feature": "state__macro__dollar_momentum_12m_lag1",
        "primary_or_secondary": "secondary sensitivity only",
        "revision_warning": "Financial series; one-month conservative lag",
    },
    "T10Y2Y": {
        "description": "10-Year minus 2-Year Treasury yield spread",
        "frequency": "daily",
        "transform": "month-end level",
        "publication_lag_months": 1,
        "feature": "state__macro__yield_curve_lag1",
        "primary_or_secondary": "secondary sensitivity only",
        "revision_warning": "Financial series; one-month conservative lag",
    },
    "DGS3MO": {
        "description": "3-Month Treasury constant-maturity rate",
        "frequency": "daily",
        "transform": "month-end annual percentage yield divided by 1200",
        "publication_lag_months": 0,
        "feature": "cash return proxy for next holding month",
        "primary_or_secondary": "risk-free proxy",
        "revision_warning": "Yield-based monthly cash-return approximation",
    },
}

def download_fred_csv(series_id):
    url = (
        "https://fred.stlouisfed.org/graph/"
        f"fredgraph.csv?id={series_id}"
    )

    request = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urllib.request.urlopen(request, timeout=60) as response:
        text = response.read().decode("utf-8")

    data = pd.read_csv(StringIO(text))

    if data.shape[1] != 2:
        raise RuntimeError(
            f"{series_id}: unexpected FRED shape {data.shape}"
        )

    data.columns = ["date", series_id]
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data[series_id] = pd.to_numeric(
        data[series_id], errors="coerce"
    )

    data = (
        data.dropna(subset=["date"])
        .set_index("date")
        .sort_index()
    )

    return data[[series_id]]

fred_raw = {}
fred_failures = []

for series_id in FRED_SPEC:
    try:
        fred_raw[series_id] = download_fred_csv(series_id)
        print(
            f"FRED {series_id}:",
            fred_raw[series_id].index.min().date(),
            "to",
            fred_raw[series_id].index.max().date(),
            "| valid:",
            int(fred_raw[series_id][series_id].notna().sum())
        )
    except Exception as exc:
        fred_failures.append(
            {
                "series_id": series_id,
                "error": repr(exc),
            }
        )
        print(f"FRED DOWNLOAD FAILED — {series_id}: {exc}")

macro_download_ok = len(fred_failures) == 0

pd.DataFrame(
    fred_failures,
    columns=["series_id", "error"]
).to_csv(
    OUTDIR / "fred_download_failures.csv",
    index=False
)

# ------------------------------------------------------------
# 4. Construct secondary macro features with explicit lags
# ------------------------------------------------------------

macro_features = pd.DataFrame()
cash_monthly_rate = pd.Series(dtype=float)

if macro_download_ok:
    fred_monthly = {}

    for series_id, frame in fred_raw.items():
        fred_monthly[series_id] = (
            frame[series_id]
            .resample("ME")
            .last()
        )

    macro_features = pd.DataFrame(
        index=pd.date_range(
            min(s.index.min() for s in fred_monthly.values()),
            max(s.index.max() for s in fred_monthly.values()),
            freq="ME"
        )
    )

    # The transformation is calculated first and then shifted.
    # Therefore, the month-t state never receives the raw
    # month-t macro observation for these four variables.
    macro_features[
        "state__macro__cpi_inflation_yoy_lag1"
    ] = (
        fred_monthly["CPIAUCSL"]
        .pct_change(12, fill_method=None)
        .mul(100.0)
        .shift(1)
    )

    macro_features[
        "state__macro__industrial_production_yoy_lag1"
    ] = (
        fred_monthly["INDPRO"]
        .pct_change(12, fill_method=None)
        .mul(100.0)
        .shift(1)
    )

    macro_features[
        "state__macro__dollar_momentum_12m_lag1"
    ] = (
        fred_monthly["DTWEXBGS"]
        .pct_change(12, fill_method=None)
        .mul(100.0)
        .shift(1)
    )

    macro_features[
        "state__macro__yield_curve_lag1"
    ] = fred_monthly["T10Y2Y"].shift(1)

    # At month-end t, this yield is observable and is used as
    # a simple approximation for cash earned during month t+1.
    cash_monthly_rate = (
        fred_monthly["DGS3MO"] / 100.0 / 12.0
    ).rename("target_next__cash_proxy")

    macro_features.to_parquet(
        OUTDIR / "secondary_macro_features_lagged.parquet"
    )

    cash_monthly_rate.to_frame().to_parquet(
        OUTDIR / "cash_rate_proxy_monthly.parquet"
    )

# ------------------------------------------------------------
# 5. Create exact next-month targets
#
# Row indexed by state month t:
#   features use information through t
#   target_month = t + 1
#   target return = observed return in t + 1
# ------------------------------------------------------------

targets = pd.DataFrame(index=returns.index)

for ticker in ASSETS:
    targets[f"target_next__{ticker.lower()}"] = (
        returns[ticker].shift(-1)
    )

for ticker in BENCHMARKS:
    targets[f"benchmark_next__{ticker.lower()}"] = (
        returns[ticker].shift(-1)
    )

targets["target_month"] = (
    targets.index.to_period("M") + 1
).to_timestamp("M")

# A month-t Treasury yield is used as the cash proxy for t+1.
if macro_download_ok:
    targets["target_next__cash_proxy"] = (
        cash_monthly_rate.reindex(targets.index)
    )

# ------------------------------------------------------------
# 6. Build the primary model-ready panel
# ------------------------------------------------------------

primary = market_features.join(targets, how="inner")

required_primary = (
    list(market_features.columns)
    + [f"target_next__{x.lower()}" for x in ASSETS]
    + [f"benchmark_next__{x.lower()}" for x in BENCHMARKS]
    + ["target_month"]
)

if macro_download_ok:
    required_primary.append("target_next__cash_proxy")

primary = primary.dropna(
    subset=required_primary
).copy()

primary.index.name = "state_month"

# Freeze the final available target month.
primary = primary.loc[
    primary["target_month"] <= FINAL_TARGET_END
].copy()

primary["sample_split"] = np.select(
    [
        primary["target_month"] <= TRAIN_TARGET_END,
        primary["target_month"] >= OOS_TARGET_START,
    ],
    [
        "TRAIN",
        "OOS",
    ],
    default="UNASSIGNED"
)

if primary["sample_split"].eq("UNASSIGNED").any():
    raise RuntimeError("Unassigned target months exist.")

primary.to_parquet(
    OUTDIR / "model_ready_primary_market_only.parquet"
)

# ------------------------------------------------------------
# 7. Secondary model-ready panel with lagged macro variables
# ------------------------------------------------------------

secondary = pd.DataFrame()

if macro_download_ok:
    secondary = (
        market_features
        .join(macro_features, how="inner")
        .join(targets, how="inner")
    )

    secondary_required = (
        list(market_features.columns)
        + list(macro_features.columns)
        + [f"target_next__{x.lower()}" for x in ASSETS]
        + [f"benchmark_next__{x.lower()}" for x in BENCHMARKS]
        + [
            "target_next__cash_proxy",
            "target_month",
        ]
    )

    secondary = secondary.dropna(
        subset=secondary_required
    ).copy()

    secondary = secondary.loc[
        secondary["target_month"] <= FINAL_TARGET_END
    ].copy()

    secondary["sample_split"] = np.select(
        [
            secondary["target_month"] <= TRAIN_TARGET_END,
            secondary["target_month"] >= OOS_TARGET_START,
        ],
        [
            "TRAIN",
            "OOS",
        ],
        default="UNASSIGNED"
    )

    secondary.to_parquet(
        OUTDIR / "model_ready_secondary_with_macro.parquet"
    )

# ------------------------------------------------------------
# 8. Exact sample timeline
# ------------------------------------------------------------

timeline_rows = []

for label, frame, feature_count in [
    ("Primary: market-only", primary, len(market_features.columns)),
    (
        "Secondary: market + lagged macro",
        secondary,
        (
            len(market_features.columns)
            + len(macro_features.columns)
            if not secondary.empty else np.nan
        )
    ),
]:
    if frame.empty:
        timeline_rows.append(
            {
                "design": label,
                "status": "NOT AVAILABLE",
                "feature_count": feature_count,
            }
        )
        continue

    train = frame.loc[frame["sample_split"].eq("TRAIN")]
    oos = frame.loc[frame["sample_split"].eq("OOS")]

    timeline_rows.append(
        {
            "design": label,
            "status": "AVAILABLE",
            "feature_count": feature_count,
            "first_state_month": frame.index.min(),
            "first_target_month": frame["target_month"].min(),
            "last_state_month": frame.index.max(),
            "last_target_month": frame["target_month"].max(),
            "total_rows": len(frame),
            "training_rows": len(train),
            "training_first_target": train["target_month"].min(),
            "training_last_target": train["target_month"].max(),
            "oos_rows": len(oos),
            "oos_first_target": oos["target_month"].min(),
            "oos_last_target": oos["target_month"].max(),
            "training_rows_per_feature": (
                len(train) / feature_count
                if feature_count else np.nan
            ),
        }
    )

timeline = pd.DataFrame(timeline_rows)
timeline.to_csv(
    OUTDIR / "exact_state_target_timeline.csv",
    index=False
)

print("\n" + "=" * 100)
print("A. EXACT STATE/TARGET TIMELINE")
print("=" * 100)
display(timeline)

# ------------------------------------------------------------
# 9. Train-only normalization audit
#
# This is only an audit. Each future walk-forward fold must fit
# its own scaler using that fold's training observations.
# ------------------------------------------------------------

feature_columns = list(market_features.columns)

train_mask = primary["sample_split"].eq("TRAIN")
train_features = primary.loc[train_mask, feature_columns]

train_mean = train_features.mean()
train_std = train_features.std(ddof=0)

zero_variance = train_std.eq(0)

if zero_variance.any():
    raise RuntimeError(
        "Zero-variance features: "
        f"{train_std.index[zero_variance].tolist()}"
    )

scaled_features = (
    primary[feature_columns] - train_mean
) / train_std

scaler = pd.DataFrame(
    {
        "feature": feature_columns,
        "training_mean": train_mean.values,
        "training_std_ddof0": train_std.values,
        "fit_first_state": train_features.index.min(),
        "fit_last_state": train_features.index.max(),
        "fit_last_target": primary.loc[
            train_mask, "target_month"
        ].max(),
        "fit_rows": len(train_features),
    }
)

scaler.to_csv(
    OUTDIR / "primary_train_only_scaler.csv",
    index=False
)

scaled_train_mean_max = (
    scaled_features.loc[train_mask].mean().abs().max()
)

scaled_train_std_error_max = (
    (
        scaled_features.loc[train_mask].std(ddof=0)
        - 1.0
    )
    .abs()
    .max()
)

# ------------------------------------------------------------
# 10. Explicit target-alignment tests
# ------------------------------------------------------------

alignment_rows = []

for ticker in ASSETS:
    target_column = f"target_next__{ticker.lower()}"

    expected = returns[ticker].shift(-1).reindex(primary.index)
    actual = primary[target_column]

    max_error = (expected - actual).abs().max()

    alignment_rows.append(
        {
            "test": f"{ticker} target equals next-month return",
            "maximum_absolute_error": max_error,
            "status": "PASS" if max_error <= 1e-15 else "FAIL",
        }
    )

for ticker in BENCHMARKS:
    target_column = f"benchmark_next__{ticker.lower()}"

    expected = returns[ticker].shift(-1).reindex(primary.index)
    actual = primary[target_column]

    max_error = (expected - actual).abs().max()

    alignment_rows.append(
        {
            "test": f"{ticker} benchmark equals next-month return",
            "maximum_absolute_error": max_error,
            "status": "PASS" if max_error <= 1e-15 else "FAIL",
        }
    )

alignment = pd.DataFrame(alignment_rows)

# ------------------------------------------------------------
# 11. Future-perturbation leakage test
#
# Change every asset return after 2018-12 by a huge amount.
# Features dated through 2018-12 must remain identical.
# ------------------------------------------------------------

perturbed_returns = returns[ASSETS].copy()

future_mask = (
    perturbed_returns.index > pd.Timestamp("2018-12-31")
)

# Deterministic, large perturbation.
perturbed_returns.loc[future_mask, :] = (
    perturbed_returns.loc[future_mask, :] + 5.0
)

perturbed_features = build_market_features(perturbed_returns)

comparison_dates = market_features.index[
    market_features.index <= pd.Timestamp("2018-12-31")
]

future_perturbation_max_error = (
    market_features.loc[comparison_dates]
    - perturbed_features.loc[comparison_dates]
).abs().max().max()

# ------------------------------------------------------------
# 12. Macro lag verification
# ------------------------------------------------------------

macro_lag_rows = []

if macro_download_ok:
    transformed_unlagged = {
        "state__macro__cpi_inflation_yoy_lag1": (
            fred_monthly["CPIAUCSL"]
            .pct_change(12, fill_method=None)
            .mul(100.0)
        ),
        "state__macro__industrial_production_yoy_lag1": (
            fred_monthly["INDPRO"]
            .pct_change(12, fill_method=None)
            .mul(100.0)
        ),
        "state__macro__dollar_momentum_12m_lag1": (
            fred_monthly["DTWEXBGS"]
            .pct_change(12, fill_method=None)
            .mul(100.0)
        ),
        "state__macro__yield_curve_lag1": (
            fred_monthly["T10Y2Y"]
        ),
    }

    for column, unlagged in transformed_unlagged.items():
        expected = unlagged.shift(1).reindex(macro_features.index)
        actual = macro_features[column]

        valid = expected.notna() & actual.notna()

        maximum_error = (
            (expected[valid] - actual[valid]).abs().max()
            if valid.any() else np.nan
        )

        macro_lag_rows.append(
            {
                "feature": column,
                "required_lag_months": 1,
                "valid_comparisons": int(valid.sum()),
                "maximum_absolute_error": maximum_error,
                "status": (
                    "PASS"
                    if pd.notna(maximum_error)
                    and maximum_error <= 1e-15
                    else "FAIL"
                ),
            }
        )

macro_lag_audit = pd.DataFrame(
    macro_lag_rows,
    columns=[
        "feature",
        "required_lag_months",
        "valid_comparisons",
        "maximum_absolute_error",
        "status",
    ]
)

macro_lag_audit.to_csv(
    OUTDIR / "macro_publication_lag_audit.csv",
    index=False
)

# ------------------------------------------------------------
# 13. Combined leakage gate
# ------------------------------------------------------------

target_columns = [
    col for col in primary.columns
    if col.startswith("target_next__")
    or col.startswith("benchmark_next__")
]

feature_name_target_overlap = sorted(
    set(feature_columns).intersection(target_columns)
)

leakage_tests = pd.DataFrame(
    [
        {
            "test": "No target or benchmark column used as a feature",
            "value": len(feature_name_target_overlap),
            "tolerance": 0,
            "status": (
                "PASS"
                if len(feature_name_target_overlap) == 0
                else "FAIL"
            ),
            "detail": str(feature_name_target_overlap),
        },
        {
            "test": "All asset targets equal exact t+1 returns",
            "value": alignment[
                "maximum_absolute_error"
            ].max(),
            "tolerance": 1e-15,
            "status": (
                "PASS"
                if alignment["status"].eq("PASS").all()
                else "FAIL"
            ),
            "detail": "See target_alignment_audit.csv",
        },
        {
            "test": "Future-return perturbation cannot change past features",
            "value": future_perturbation_max_error,
            "tolerance": 1e-15,
            "status": (
                "PASS"
                if future_perturbation_max_error <= 1e-15
                else "FAIL"
            ),
            "detail": (
                "Returns after 2018-12 were increased by 5.0"
            ),
        },
        {
            "test": "Primary scaler fitted only through training targets",
            "value": str(
                primary.loc[
                    train_mask, "target_month"
                ].max().date()
            ),
            "tolerance": str(TRAIN_TARGET_END.date()),
            "status": (
                "PASS"
                if primary.loc[
                    train_mask, "target_month"
                ].max() == TRAIN_TARGET_END
                else "FAIL"
            ),
            "detail": (
                f"Fit rows: {len(train_features)}"
            ),
        },
        {
            "test": "Scaled training-feature mean approximately zero",
            "value": scaled_train_mean_max,
            "tolerance": 1e-12,
            "status": (
                "PASS"
                if scaled_train_mean_max <= 1e-12
                else "FAIL"
            ),
            "detail": "Maximum absolute training mean",
        },
        {
            "test": "Scaled training-feature standard deviation equals one",
            "value": scaled_train_std_error_max,
            "tolerance": 1e-12,
            "status": (
                "PASS"
                if scaled_train_std_error_max <= 1e-12
                else "FAIL"
            ),
            "detail": "Maximum absolute standard-deviation error",
        },
        {
            "test": "Macro variables excluded from primary specification",
            "value": 0,
            "tolerance": 0,
            "status": "PASS",
            "detail": (
                "Lagged macro variables retained only for sensitivity"
            ),
        },
        {
            "test": "All four secondary macro features have explicit lag",
            "value": (
                int(macro_lag_audit["status"].eq("PASS").sum())
                if not macro_lag_audit.empty else 0
            ),
            "tolerance": 4,
            "status": (
                "PASS"
                if not macro_lag_audit.empty
                and macro_lag_audit["status"].eq("PASS").all()
                else "NOT RUN"
            ),
            "detail": (
                "Current FRED vintage remains revision-sensitive; "
                "therefore macro specification is secondary only"
            ),
        },
    ]
)

alignment.to_csv(
    OUTDIR / "target_alignment_audit.csv",
    index=False
)

leakage_tests.to_csv(
    OUTDIR / "leakage_test_results.csv",
    index=False
)

print("\n" + "=" * 100)
print("B. TARGET ALIGNMENT")
print("=" * 100)
display(alignment)

print("\n" + "=" * 100)
print("C. MACRO PUBLICATION-LAG AUDIT")
print("=" * 100)
display(macro_lag_audit)

print("\n" + "=" * 100)
print("D. LEAKAGE TESTS")
print("=" * 100)
display(leakage_tests)

# Fail immediately if a mandatory primary test fails.
mandatory = leakage_tests.loc[
    ~leakage_tests["status"].eq("NOT RUN")
]

if mandatory["status"].eq("FAIL").any():
    failed = mandatory.loc[
        mandatory["status"].eq("FAIL"), "test"
    ].tolist()
    raise RuntimeError(f"Mandatory leakage tests failed: {failed}")

# ------------------------------------------------------------
# 14. Feature dictionary
# ------------------------------------------------------------

dictionary_rows = []

for ticker in ASSETS:
    dictionary_rows.extend(
        [
            {
                "column": f"state__{ticker.lower()}__momentum_6m",
                "role": "primary state feature",
                "source": f"CRSP msf_v2.mthret, PERMNO {dict(DBE=91709, GLD=90448, DBA=91712, DBB=91715)[ticker]}",
                "definition": "Compounded return over months t-5 through t",
                "information_available": "Month-end t",
            },
            {
                "column": f"state__{ticker.lower()}__momentum_12m",
                "role": "primary state feature",
                "source": f"CRSP msf_v2.mthret, PERMNO {dict(DBE=91709, GLD=90448, DBA=91712, DBB=91715)[ticker]}",
                "definition": "Compounded return over months t-11 through t",
                "information_available": "Month-end t",
            },
            {
                "column": f"state__{ticker.lower()}__volatility_12m",
                "role": "primary state feature",
                "source": f"CRSP msf_v2.mthret, PERMNO {dict(DBE=91709, GLD=90448, DBA=91712, DBB=91715)[ticker]}",
                "definition": "Sample standard deviation of returns from t-11 through t",
                "information_available": "Month-end t",
            },
        ]
    )

for series_id, spec in FRED_SPEC.items():
    dictionary_rows.append(
        {
            "column": spec["feature"],
            "role": spec["primary_or_secondary"],
            "source": f"FRED {series_id}",
            "definition": spec["transform"],
            "information_available": (
                f"Shifted {spec['publication_lag_months']} month(s); "
                f"{spec['revision_warning']}"
            ),
        }
    )

feature_dictionary = pd.DataFrame(dictionary_rows)
feature_dictionary.to_csv(
    OUTDIR / "feature_and_timing_dictionary.csv",
    index=False
)

# ------------------------------------------------------------
# 15. Hash updated outputs
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

hash_rows = []

for path in sorted(OUTDIR.iterdir()):
    if path.is_file() and path.name != "audit_file_hashes_stage3.csv":
        hash_rows.append(
            {
                "filename": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

pd.DataFrame(hash_rows).to_csv(
    OUTDIR / "audit_file_hashes_stage3.csv",
    index=False
)

print("\n" + "=" * 100)
print("FEATURE AND LEAKAGE AUDIT COMPLETE")
print("=" * 100)
print("Primary design: 12 market-only features")
print("Secondary design: 12 market + 4 explicitly lagged macro features")
print("Primary macro revision exposure: NONE")
print("Output directory:", OUTDIR)
print()
print("PASTE BACK SECTIONS A, C, AND D.")

In [ ]:
# ============================================================
# REVIEWER AUDIT 4B — COMPLETE STANDALONE CONTINUATION
#
# Primary sample ends at the last jointly verified month:
# December 2024.
#
# No WRDS reconnection is required.
# ============================================================

from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 300)

# ------------------------------------------------------------
# 0. Configuration and reload
# ------------------------------------------------------------

OUTDIR = Path.home() / "gsci_reviewer_audit_20260722_205114"

RETURNS_PATH = (
    OUTDIR / "verified_monthly_total_return_panel_NO_FILL.parquet"
)

PRIMARY_PATH = (
    OUTDIR / "model_ready_primary_market_only.parquet"
)

CASH_PATH = (
    OUTDIR / "crsp_30day_tbill_monthly_return.parquet"
)

ASSETS = ["DBE", "GLD", "DBA", "DBB"]
BENCHMARKS = ["GSG", "DBC"]
PORTFOLIO_ASSETS = ASSETS + ["CASH"]

TRAIN_TARGET_END = pd.Timestamp("2018-12-31")
OOS_TARGET_START = pd.Timestamp("2019-01-31")

PRIMARY_COST_BPS = 10
COST_GRID_BPS = [0, 5, 10, 25]

for required_path in [
    RETURNS_PATH,
    PRIMARY_PATH,
    CASH_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

returns = pd.read_parquet(RETURNS_PATH).sort_index()
returns.index = pd.to_datetime(returns.index)
returns.index.name = "return_month"

primary = pd.read_parquet(PRIMARY_PATH).sort_index()
primary.index = pd.to_datetime(primary.index)
primary.index.name = "state_month"
primary["target_month"] = pd.to_datetime(
    primary["target_month"]
)

cash_frame = pd.read_parquet(CASH_PATH).sort_index()
cash_frame.index = pd.to_datetime(cash_frame.index)

if cash_frame.shape[1] != 1:
    raise RuntimeError(
        f"Expected one cash-return column; found {cash_frame.columns.tolist()}"
    )

cash_monthly = pd.to_numeric(
    cash_frame.iloc[:, 0],
    errors="coerce"
).dropna()

cash_monthly.name = "cash_return"

# ------------------------------------------------------------
# 1. Freeze the common verified sample
# ------------------------------------------------------------

asset_target_end = primary["target_month"].max()
cash_target_end = cash_monthly.index.max()

FINAL_TARGET_END = min(
    asset_target_end,
    cash_target_end
)

if FINAL_TARGET_END != pd.Timestamp("2024-12-31"):
    print(
        "WARNING: Automatically determined final target month:",
        FINAL_TARGET_END
    )

original_rows = len(primary)

primary = primary.loc[
    primary["target_month"] <= FINAL_TARGET_END
].copy()

dropped_rows = original_rows - len(primary)

sample_end_decision = pd.DataFrame(
    [
        {
            "asset_return_end": asset_target_end,
            "crsp_cash_return_end": cash_target_end,
            "frozen_primary_target_end": FINAL_TARGET_END,
            "rows_excluded_after_common_end": dropped_rows,
            "missing_cash_imputed": False,
            "decision": (
                "Primary analysis ends at last month jointly "
                "covered by asset, benchmark, and CRSP cash returns"
            ),
        }
    ]
)

sample_end_decision.to_csv(
    OUTDIR / "primary_sample_end_decision.csv",
    index=False
)

print("=" * 110)
print("PRIMARY COMMON-SAMPLE DECISION")
print("=" * 110)
display(sample_end_decision)

# ------------------------------------------------------------
# 2. Attach exact contemporaneous cash returns
# ------------------------------------------------------------

primary["target_next__cash"] = (
    cash_monthly.reindex(
        primary["target_month"]
    ).to_numpy()
)

if primary["target_next__cash"].isna().any():
    missing_dates = primary.loc[
        primary["target_next__cash"].isna(),
        "target_month"
    ].tolist()

    raise RuntimeError(
        f"Cash returns still missing for: {missing_dates}"
    )

primary.to_parquet(
    OUTDIR / "model_ready_primary_with_crsp_cash_2008_2024.parquet"
)

# ------------------------------------------------------------
# 3. Build all transparent strategy weights
# ------------------------------------------------------------

def zero_weights(index):
    return pd.DataFrame(
        0.0,
        index=index,
        columns=PORTFOLIO_ASSETS
    )

def build_strategy_weights(state_panel):
    output = {}

    # Equal weight
    weights = zero_weights(state_panel.index)
    weights[ASSETS] = 0.25
    output["equal_weight"] = weights

    # Inverse 12-month volatility
    weights = zero_weights(state_panel.index)

    inverse_volatility = pd.DataFrame(
        {
            ticker: (
                1.0
                / state_panel[
                    f"state__{ticker.lower()}__volatility_12m"
                ].clip(lower=1e-8)
            )
            for ticker in ASSETS
        },
        index=state_panel.index
    )

    weights[ASSETS] = inverse_volatility.div(
        inverse_volatility.sum(axis=1),
        axis=0
    )

    output["inverse_vol_12m"] = weights

    # Top-two 6-month momentum, inverse-volatility weighted
    weights = zero_weights(state_panel.index)

    for state_month, row in state_panel.iterrows():
        momentum = pd.Series(
            {
                ticker: row[
                    f"state__{ticker.lower()}__momentum_6m"
                ]
                for ticker in ASSETS
            }
        )

        selected = momentum.nlargest(2).index.tolist()

        inverse_selected_volatility = pd.Series(
            {
                ticker: (
                    1.0
                    / max(
                        row[
                            f"state__{ticker.lower()}__volatility_12m"
                        ],
                        1e-8
                    )
                )
                for ticker in selected
            }
        )

        allocation = (
            inverse_selected_volatility
            / inverse_selected_volatility.sum()
        )

        weights.loc[
            state_month,
            selected
        ] = allocation.values

    output["momentum_6m_top2_invvol"] = weights

    # Top-two 12-month momentum, inverse-volatility weighted
    weights = zero_weights(state_panel.index)

    for state_month, row in state_panel.iterrows():
        momentum = pd.Series(
            {
                ticker: row[
                    f"state__{ticker.lower()}__momentum_12m"
                ]
                for ticker in ASSETS
            }
        )

        selected = momentum.nlargest(2).index.tolist()

        inverse_selected_volatility = pd.Series(
            {
                ticker: (
                    1.0
                    / max(
                        row[
                            f"state__{ticker.lower()}__volatility_12m"
                        ],
                        1e-8
                    )
                )
                for ticker in selected
            }
        )

        allocation = (
            inverse_selected_volatility
            / inverse_selected_volatility.sum()
        )

        weights.loc[
            state_month,
            selected
        ] = allocation.values

    output["momentum_12m_top2_invvol"] = weights

    # Best 12-month momentum asset, otherwise cash
    weights = zero_weights(state_panel.index)

    for state_month, row in state_panel.iterrows():
        momentum = pd.Series(
            {
                ticker: row[
                    f"state__{ticker.lower()}__momentum_12m"
                ]
                for ticker in ASSETS
            }
        )

        winner = momentum.idxmax()

        if momentum.loc[winner] > 0:
            weights.loc[state_month, winner] = 1.0
        else:
            weights.loc[state_month, "CASH"] = 1.0

    output["absolute_momentum_12m_cash"] = weights

    return output

strategy_weights = build_strategy_weights(primary)

# ------------------------------------------------------------
# 4. Validate weights
# ------------------------------------------------------------

weight_audit_rows = []

for strategy, weights in strategy_weights.items():
    sum_error = (
        weights.sum(axis=1) - 1.0
    ).abs().max()

    minimum_weight = weights.min().min()
    maximum_weight = weights.max().max()
    missing_values = int(weights.isna().sum().sum())

    status = (
        "PASS"
        if (
            sum_error <= 1e-12
            and minimum_weight >= -1e-12
            and maximum_weight <= 1.0 + 1e-12
            and missing_values == 0
        )
        else "FAIL"
    )

    weight_audit_rows.append(
        {
            "strategy": strategy,
            "maximum_weight_sum_error": sum_error,
            "minimum_weight": minimum_weight,
            "maximum_weight": maximum_weight,
            "missing_values": missing_values,
            "status": status,
        }
    )

weight_audit = pd.DataFrame(weight_audit_rows)

print("\n" + "=" * 110)
print("A. WEIGHT VALIDATION")
print("=" * 110)
display(weight_audit)

if not weight_audit["status"].eq("PASS").all():
    raise RuntimeError("Weight validation failed.")

# ------------------------------------------------------------
# 5. Create target-return and state-return matrices
# ------------------------------------------------------------

target_returns = pd.DataFrame(
    index=primary.index,
    columns=PORTFOLIO_ASSETS,
    dtype=float
)

for ticker in ASSETS:
    target_returns[ticker] = primary[
        f"target_next__{ticker.lower()}"
    ]

target_returns["CASH"] = primary[
    "target_next__cash"
]

state_month_returns = returns[ASSETS].reindex(
    primary.index
).copy()

state_month_returns["CASH"] = cash_monthly.reindex(
    primary.index
)

if target_returns.isna().any().any():
    raise RuntimeError(
        "Target-return matrix contains missing values."
    )

if state_month_returns.isna().any().any():
    missing = state_month_returns.isna().sum()
    raise RuntimeError(
        "State-return matrix contains missing values:\n"
        f"{missing[missing > 0]}"
    )

# ------------------------------------------------------------
# 6. Drift-adjusted turnover
# ------------------------------------------------------------

def drift_adjusted_turnover(
    weights,
    realized_state_returns
):
    turnover = pd.Series(
        np.nan,
        index=weights.index,
        name="turnover"
    )

    detail_rows = []

    # The first row is far before the OOS evaluation.
    turnover.iloc[0] = 0.0

    for position in range(1, len(weights)):
        state_month = weights.index[position]
        previous_state = weights.index[position - 1]

        previous_target = weights.loc[previous_state]
        current_target = weights.loc[state_month]

        realized_returns = realized_state_returns.loc[
            state_month
        ]

        end_values = (
            previous_target
            * (1.0 + realized_returns)
        )

        portfolio_end_value = end_values.sum()

        if portfolio_end_value <= 0:
            raise RuntimeError(
                f"Nonpositive portfolio value at {state_month}"
            )

        drifted_previous = (
            end_values / portfolio_end_value
        )

        current_turnover = (
            0.5
            * (current_target - drifted_previous).abs().sum()
        )

        turnover.loc[state_month] = current_turnover

        detail_rows.append(
            {
                "state_month": state_month,
                "previous_state_month": previous_state,
                **{
                    f"previous_target__{asset}":
                        previous_target[asset]
                    for asset in PORTFOLIO_ASSETS
                },
                **{
                    f"state_return__{asset}":
                        realized_returns[asset]
                    for asset in PORTFOLIO_ASSETS
                },
                **{
                    f"drifted_previous__{asset}":
                        drifted_previous[asset]
                    for asset in PORTFOLIO_ASSETS
                },
                **{
                    f"current_target__{asset}":
                        current_target[asset]
                    for asset in PORTFOLIO_ASSETS
                },
                "turnover": current_turnover,
            }
        )

    return turnover, pd.DataFrame(detail_rows)

strategy_turnover = {}
strategy_drift_details = {}

for strategy, weights in strategy_weights.items():
    turnover, details = drift_adjusted_turnover(
        weights,
        state_month_returns
    )

    strategy_turnover[strategy] = turnover
    strategy_drift_details[strategy] = details

    if turnover.min() < -1e-12:
        raise RuntimeError(
            f"{strategy}: negative turnover."
        )

    if turnover.max() > 1.0 + 1e-12:
        raise RuntimeError(
            f"{strategy}: turnover exceeds 1.0."
        )

# ------------------------------------------------------------
# 7. Calculate gross and cost-adjusted returns
# ------------------------------------------------------------

strategy_gross = {}
strategy_net = {
    cost_bps: {} for cost_bps in COST_GRID_BPS
}

for strategy, weights in strategy_weights.items():
    gross = (
        weights[PORTFOLIO_ASSETS]
        * target_returns[PORTFOLIO_ASSETS]
    ).sum(axis=1)

    strategy_gross[strategy] = gross

    for cost_bps in COST_GRID_BPS:
        strategy_net[cost_bps][strategy] = (
            gross
            - (
                cost_bps / 10_000.0
                * strategy_turnover[strategy]
            )
        )

passive_returns = {
    "GSG_buy_and_hold":
        primary["benchmark_next__gsg"],
    "DBC_buy_and_hold":
        primary["benchmark_next__dbc"],
    "cash_30day_tbill":
        primary["target_next__cash"],
}

# ------------------------------------------------------------
# 8. Correct performance metrics
# ------------------------------------------------------------

def maximum_drawdown(monthly_returns):
    monthly_returns = pd.Series(
        monthly_returns,
        dtype=float
    ).dropna()

    wealth = np.concatenate(
        [
            np.array([1.0]),
            np.cumprod(
                1.0 + monthly_returns.to_numpy()
            )
        ]
    )

    running_peak = np.maximum.accumulate(wealth)
    drawdown = wealth / running_peak - 1.0

    return float(drawdown.min())

def performance_metrics(
    monthly_returns,
    cash_returns,
    turnover=None
):
    monthly_returns = pd.Series(
        monthly_returns,
        dtype=float
    ).dropna()

    cash_returns = pd.Series(
        cash_returns,
        dtype=float
    ).reindex(monthly_returns.index)

    if cash_returns.isna().any():
        raise RuntimeError(
            "Missing cash during performance calculation."
        )

    if (monthly_returns <= -1.0).any():
        raise RuntimeError(
            "Portfolio return <= -100% detected."
        )

    months = len(monthly_returns)

    ending_wealth = float(
        np.prod(
            1.0 + monthly_returns.to_numpy()
        )
    )

    total_return = ending_wealth - 1.0

    cagr = (
        ending_wealth ** (12.0 / months)
        - 1.0
    )

    arithmetic_annual_return = (
        12.0 * monthly_returns.mean()
    )

    annual_volatility = (
        np.sqrt(12.0)
        * monthly_returns.std(ddof=1)
    )

    excess_returns = (
        monthly_returns - cash_returns
    )

    excess_standard_deviation = (
        excess_returns.std(ddof=1)
    )

    excess_sharpe = (
        np.sqrt(12.0)
        * excess_returns.mean()
        / excess_standard_deviation
        if excess_standard_deviation > 0
        else np.nan
    )

    max_drawdown = maximum_drawdown(
        monthly_returns
    )

    calmar = (
        cagr / abs(max_drawdown)
        if max_drawdown < 0
        else np.nan
    )

    if turnover is None:
        average_monthly_turnover = 0.0
    else:
        turnover = pd.Series(
            turnover,
            dtype=float
        ).reindex(monthly_returns.index)

        if turnover.isna().any():
            raise RuntimeError(
                "Missing turnover in evaluation period."
            )

        average_monthly_turnover = turnover.mean()

    return {
        "months": months,
        "total_return": total_return,
        "cagr": cagr,
        "arithmetic_annual_return":
            arithmetic_annual_return,
        "annual_volatility": annual_volatility,
        "excess_sharpe": excess_sharpe,
        "maximum_drawdown": max_drawdown,
        "calmar": calmar,
        "monthly_hit_rate_over_cash": float(
            (monthly_returns > cash_returns).mean()
        ),
        "average_monthly_turnover":
            average_monthly_turnover,
        "annualized_turnover":
            12.0 * average_monthly_turnover,
    }

# ------------------------------------------------------------
# 9. Freeze exact OOS sample
# ------------------------------------------------------------

oos_mask = primary["target_month"].between(
    OOS_TARGET_START,
    FINAL_TARGET_END
)

oos_index = primary.index[oos_mask]

expected_oos_months = len(
    pd.date_range(
        OOS_TARGET_START,
        FINAL_TARGET_END,
        freq="ME"
    )
)

if len(oos_index) != expected_oos_months:
    raise RuntimeError(
        f"Expected {expected_oos_months} OOS months; "
        f"found {len(oos_index)}."
    )

oos_cash = primary.loc[
    oos_index,
    "target_next__cash"
]

timeline = pd.DataFrame(
    [
        {
            "first_training_state":
                primary.loc[
                    primary["target_month"]
                    <= TRAIN_TARGET_END
                ].index.min(),
            "first_training_target":
                primary.loc[
                    primary["target_month"]
                    <= TRAIN_TARGET_END,
                    "target_month"
                ].min(),
            "last_training_target": TRAIN_TARGET_END,
            "training_rows": int(
                (
                    primary["target_month"]
                    <= TRAIN_TARGET_END
                ).sum()
            ),
            "first_oos_state": oos_index.min(),
            "first_oos_target":
                primary.loc[
                    oos_index,
                    "target_month"
                ].min(),
            "last_oos_state": oos_index.max(),
            "last_oos_target":
                primary.loc[
                    oos_index,
                    "target_month"
                ].max(),
            "oos_rows": len(oos_index),
        }
    ]
)

timeline.to_csv(
    OUTDIR / "final_primary_timeline_2008_2024.csv",
    index=False
)

print("\n" + "=" * 110)
print("B. FINAL TRAIN/OOS TIMELINE")
print("=" * 110)
display(timeline)

# ------------------------------------------------------------
# 10. Evaluate all strategies and costs
# ------------------------------------------------------------

performance_rows = []

for cost_bps in COST_GRID_BPS:
    for strategy in strategy_weights:
        evaluated_return = (
            strategy_net[cost_bps][strategy]
            .loc[oos_index]
        )

        evaluated_turnover = (
            strategy_turnover[strategy]
            .loc[oos_index]
        )

        metrics = performance_metrics(
            evaluated_return,
            oos_cash,
            evaluated_turnover
        )

        metrics.update(
            {
                "strategy": strategy,
                "cost_bps": cost_bps,
                "return_type": "net",
                "oos_first_target":
                    primary.loc[
                        oos_index,
                        "target_month"
                    ].min(),
                "oos_last_target":
                    primary.loc[
                        oos_index,
                        "target_month"
                    ].max(),
            }
        )

        performance_rows.append(metrics)

    for strategy, evaluated_return_full in passive_returns.items():
        evaluated_return = (
            evaluated_return_full.loc[oos_index]
        )

        metrics = performance_metrics(
            evaluated_return,
            oos_cash,
            turnover=None
        )

        metrics.update(
            {
                "strategy": strategy,
                "cost_bps": cost_bps,
                "return_type": "passive total return",
                "oos_first_target":
                    primary.loc[
                        oos_index,
                        "target_month"
                    ].min(),
                "oos_last_target":
                    primary.loc[
                        oos_index,
                        "target_month"
                    ].max(),
            }
        )

        performance_rows.append(metrics)

performance = pd.DataFrame(performance_rows)

performance = performance[
    [
        "strategy",
        "cost_bps",
        "return_type",
        "months",
        "cagr",
        "arithmetic_annual_return",
        "annual_volatility",
        "excess_sharpe",
        "maximum_drawdown",
        "calmar",
        "total_return",
        "monthly_hit_rate_over_cash",
        "average_monthly_turnover",
        "annualized_turnover",
        "oos_first_target",
        "oos_last_target",
    ]
].sort_values(
    ["cost_bps", "excess_sharpe"],
    ascending=[True, False]
)

performance.to_csv(
    OUTDIR / "baseline_performance_all_costs_2019_2024.csv",
    index=False
)

primary_cost_table = (
    performance.loc[
        performance["cost_bps"]
        .eq(PRIMARY_COST_BPS)
    ]
    .sort_values(
        "excess_sharpe",
        ascending=False
    )
)

primary_cost_table.to_csv(
    OUTDIR / "baseline_performance_10bps_2019_2024.csv",
    index=False
)

print("\n" + "=" * 110)
print("C. PRIMARY OOS RESULTS — 10 BPS")
print("=" * 110)
display(primary_cost_table)

# ------------------------------------------------------------
# 11. Cost sensitivity
# ------------------------------------------------------------

cost_sensitivity = performance.pivot_table(
    index="strategy",
    columns="cost_bps",
    values=[
        "cagr",
        "excess_sharpe",
        "total_return"
    ],
    aggfunc="first"
)

cost_sensitivity.to_csv(
    OUTDIR / "baseline_cost_sensitivity_2019_2024.csv"
)

print("\n" + "=" * 110)
print("D. TRANSACTION-COST SENSITIVITY")
print("=" * 110)
display(cost_sensitivity)

# ------------------------------------------------------------
# 12. Worked drift-adjusted turnover example
# ------------------------------------------------------------

example_strategy = "inverse_vol_12m"

example_state = (
    strategy_turnover[example_strategy]
    .loc[oos_index]
    .idxmax()
)

example = (
    strategy_drift_details[example_strategy]
    .set_index("state_month")
    .loc[[example_state]]
)

example.to_csv(
    OUTDIR / "worked_drift_adjusted_turnover_example.csv"
)

print("\n" + "=" * 110)
print("E. WORKED TURNOVER EXAMPLE")
print("=" * 110)
display(example.T)

# ------------------------------------------------------------
# 13. Actual no-look-ahead reconstruction
# ------------------------------------------------------------

feature_columns = [
    column
    for column in primary.columns
    if column.startswith("state__")
]

perturbed_primary = primary.copy()

perturbed_primary.loc[
    perturbed_primary["target_month"]
    >= OOS_TARGET_START,
    feature_columns
] += 1000.0

perturbed_weights = build_strategy_weights(
    perturbed_primary
)

pre_oos_index = primary.index[
    primary["target_month"] <= TRAIN_TARGET_END
]

# ------------------------------------------------------------
# 14. Calculation audits
# ------------------------------------------------------------

audit_rows = []

for strategy, weights in strategy_weights.items():
    independent_gross = (
        weights[PORTFOLIO_ASSETS].to_numpy()
        * target_returns[PORTFOLIO_ASSETS].to_numpy()
    ).sum(axis=1)

    gross_error = np.max(
        np.abs(
            independent_gross
            - strategy_gross[strategy].to_numpy()
        )
    )

    audit_rows.append(
        {
            "test": f"{strategy}: weights × returns",
            "maximum_error": gross_error,
            "tolerance": 1e-15,
            "status": (
                "PASS"
                if gross_error <= 1e-15
                else "FAIL"
            ),
        }
    )

    expected_net = (
        strategy_gross[strategy]
        - (
            PRIMARY_COST_BPS / 10_000.0
            * strategy_turnover[strategy]
        )
    )

    actual_net = (
        strategy_net[
            PRIMARY_COST_BPS
        ][strategy]
    )

    net_error = (
        expected_net - actual_net
    ).abs().max()

    audit_rows.append(
        {
            "test": f"{strategy}: net-return cost identity",
            "maximum_error": net_error,
            "tolerance": 1e-15,
            "status": (
                "PASS"
                if net_error <= 1e-15
                else "FAIL"
            ),
        }
    )

    oos_return = actual_net.loc[oos_index]

    independent_cagr = (
        np.prod(
            1.0 + oos_return.to_numpy()
        )
        ** (12.0 / len(oos_return))
        - 1.0
    )

    table_cagr = primary_cost_table.loc[
        primary_cost_table["strategy"]
        .eq(strategy),
        "cagr"
    ].iloc[0]

    cagr_error = abs(
        independent_cagr - table_cagr
    )

    audit_rows.append(
        {
            "test": f"{strategy}: independent CAGR",
            "maximum_error": cagr_error,
            "tolerance": 1e-15,
            "status": (
                "PASS"
                if cagr_error <= 1e-15
                else "FAIL"
            ),
        }
    )

    lookahead_error = (
        weights.loc[pre_oos_index]
        - perturbed_weights[strategy]
        .loc[pre_oos_index]
    ).abs().max().max()

    audit_rows.append(
        {
            "test": (
                f"{strategy}: future-feature "
                "perturbation test"
            ),
            "maximum_error": lookahead_error,
            "tolerance": 1e-15,
            "status": (
                "PASS"
                if lookahead_error <= 1e-15
                else "FAIL"
            ),
        }
    )

    turnover = strategy_turnover[strategy]

    turnover_bound_error = max(
        0.0,
        float(-turnover.min()),
        float(turnover.max() - 1.0)
    )

    audit_rows.append(
        {
            "test": f"{strategy}: turnover bounds",
            "maximum_error": turnover_bound_error,
            "tolerance": 1e-12,
            "status": (
                "PASS"
                if turnover_bound_error <= 1e-12
                else "FAIL"
            ),
        }
    )

calculation_audit = pd.DataFrame(audit_rows)

calculation_audit.to_csv(
    OUTDIR / "financial_calculation_audit_2019_2024.csv",
    index=False
)

print("\n" + "=" * 110)
print("F. FINANCIAL CALCULATION AUDIT")
print("=" * 110)
display(calculation_audit)

if not calculation_audit["status"].eq("PASS").all():
    failed = calculation_audit.loc[
        calculation_audit["status"].eq("FAIL"),
        "test"
    ].tolist()

    raise RuntimeError(
        f"Calculation audit failed: {failed}"
    )

# ------------------------------------------------------------
# 15. Save weights and monthly strategy returns
# ------------------------------------------------------------

weights_long = []

for strategy, weights in strategy_weights.items():
    temporary = weights.copy()
    temporary["strategy"] = strategy
    temporary["state_month"] = temporary.index
    temporary["target_month"] = (
        primary.loc[
            temporary.index,
            "target_month"
        ].to_numpy()
    )

    weights_long.append(
        temporary.reset_index(drop=True)
    )

weights_long = pd.concat(
    weights_long,
    ignore_index=True
)

weights_long.to_parquet(
    OUTDIR / "baseline_target_weights_2008_2024.parquet",
    index=False
)

monthly_output = pd.DataFrame(
    index=primary.index
)

monthly_output["target_month"] = (
    primary["target_month"]
)

monthly_output["cash_return"] = (
    primary["target_next__cash"]
)

for strategy in strategy_weights:
    monthly_output[
        f"{strategy}__gross"
    ] = strategy_gross[strategy]

    monthly_output[
        f"{strategy}__turnover"
    ] = strategy_turnover[strategy]

    monthly_output[
        f"{strategy}__net_10bps"
    ] = strategy_net[10][strategy]

monthly_output["GSG_buy_and_hold"] = (
    primary["benchmark_next__gsg"]
)

monthly_output["DBC_buy_and_hold"] = (
    primary["benchmark_next__dbc"]
)

monthly_output.to_parquet(
    OUTDIR / "baseline_monthly_returns_2008_2024.parquet"
)

# ------------------------------------------------------------
# 16. Updated hashes
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()

hash_rows = []

for path in sorted(OUTDIR.iterdir()):
    if (
        path.is_file()
        and path.name
        != "audit_file_hashes_stage4_final.csv"
    ):
        hash_rows.append(
            {
                "filename": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

pd.DataFrame(hash_rows).to_csv(
    OUTDIR / "audit_file_hashes_stage4_final.csv",
    index=False
)

print("\n" + "=" * 110)
print("AUDIT 4 COMPLETE")
print("=" * 110)
print("Primary sample target end:", FINAL_TARGET_END.date())
print("Training targets: through 2018-12")
print("OOS targets: 2019-01 through 2024-12")
print("OOS months:", len(oos_index))
print("Cash source: CRSP mcti.t30ret")
print("Primary transaction cost:", PRIMARY_COST_BPS, "bps")
print("No 2025 cash returns were fabricated or filled.")
print("Output directory:", OUTDIR)
print()
print("PASTE BACK SECTIONS C, D, E, AND F.")

In [ ]:
# ============================================================
# REVIEWER AUDIT 5
# Fixed supervised expert selector + bootstrap inference +
# multiple-testing correction + historical robustness
# ============================================================

from pathlib import Path
import hashlib
import math

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.ensemble import GradientBoostingRegressor
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 280)
pd.set_option("display.max_rows", 300)

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

OUTDIR = Path.home() / "gsci_reviewer_audit_20260722_205114"

PRIMARY_PATH = (
    OUTDIR / "model_ready_primary_with_crsp_cash_2008_2024.parquet"
)

WEIGHTS_PATH = (
    OUTDIR / "baseline_target_weights_2008_2024.parquet"
)

MONTHLY_PATH = (
    OUTDIR / "baseline_monthly_returns_2008_2024.parquet"
)

RETURNS_PATH = (
    OUTDIR / "verified_monthly_total_return_panel_NO_FILL.parquet"
)

ASSETS = ["DBE", "GLD", "DBA", "DBB"]
PORTFOLIO_ASSETS = ASSETS + ["CASH"]

EXPERTS = [
    "equal_weight",
    "inverse_vol_12m",
    "momentum_6m_top2_invvol",
    "momentum_12m_top2_invvol",
    "absolute_momentum_12m_cash",
]

FEATURE_COLUMNS = [
    f"state__{ticker.lower()}__momentum_6m"
    for ticker in ASSETS
] + [
    f"state__{ticker.lower()}__momentum_12m"
    for ticker in ASSETS
] + [
    f"state__{ticker.lower()}__volatility_12m"
    for ticker in ASSETS
]

OOS_START = pd.Timestamp("2019-01-31")
OOS_END = pd.Timestamp("2024-12-31")

PRIMARY_COST_BPS = 10
COST_GRID_BPS = [0, 5, 10, 25]

MODEL_RANDOM_SEED = 20260722

GBM_PARAMETERS = {
    "n_estimators": 50,
    "learning_rate": 0.03,
    "max_depth": 2,
    "min_samples_leaf": 10,
    "subsample": 1.0,
    "loss": "squared_error",
}

BOOTSTRAP_REPLICATIONS = 5000
BOOTSTRAP_BLOCK_LENGTHS = [3, 6, 12]
BOOTSTRAP_SEED = 20260722

print("=" * 115)
print("SUPERVISED EXPERT SELECTOR AND INFERENCE AUDIT")
print("=" * 115)

for path in [
    PRIMARY_PATH,
    WEIGHTS_PATH,
    MONTHLY_PATH,
    RETURNS_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(path)

primary = pd.read_parquet(PRIMARY_PATH).sort_index()
primary.index = pd.to_datetime(primary.index)
primary["target_month"] = pd.to_datetime(
    primary["target_month"]
)

weights_long = pd.read_parquet(WEIGHTS_PATH)
weights_long["state_month"] = pd.to_datetime(
    weights_long["state_month"]
)
weights_long["target_month"] = pd.to_datetime(
    weights_long["target_month"]
)

baseline_monthly = pd.read_parquet(MONTHLY_PATH).sort_index()
baseline_monthly.index = pd.to_datetime(
    baseline_monthly.index
)
baseline_monthly["target_month"] = pd.to_datetime(
    baseline_monthly["target_month"]
)

raw_returns = pd.read_parquet(RETURNS_PATH).sort_index()
raw_returns.index = pd.to_datetime(raw_returns.index)

# ------------------------------------------------------------
# 1. Recover exact expert-weight panels
# ------------------------------------------------------------

expert_weights = {}

for expert in EXPERTS:
    frame = weights_long.loc[
        weights_long["strategy"].eq(expert)
    ].copy()

    if frame.empty:
        raise RuntimeError(
            f"Expert weights not found: {expert}"
        )

    frame = frame.set_index("state_month").sort_index()

    expert_weights[expert] = frame[
        PORTFOLIO_ASSETS
    ].reindex(primary.index)

    if expert_weights[expert].isna().any().any():
        raise RuntimeError(
            f"Missing weights for expert {expert}"
        )

# ------------------------------------------------------------
# 2. Reconstruct target returns and expert gross outcomes
# ------------------------------------------------------------

target_returns = pd.DataFrame(
    index=primary.index,
    columns=PORTFOLIO_ASSETS,
    dtype=float
)

for ticker in ASSETS:
    target_returns[ticker] = primary[
        f"target_next__{ticker.lower()}"
    ]

target_returns["CASH"] = primary[
    "target_next__cash"
]

expert_gross_returns = pd.DataFrame(
    index=primary.index,
    columns=EXPERTS,
    dtype=float
)

for expert in EXPERTS:
    expert_gross_returns[expert] = (
        expert_weights[expert]
        * target_returns
    ).sum(axis=1)

if expert_gross_returns.isna().any().any():
    raise RuntimeError(
        "Missing reconstructed expert gross returns."
    )

# ------------------------------------------------------------
# 3. Fixed expanding-window selector
#
# No hyperparameter tuning.
# No validation search.
# No model-family selection.
#
# For target month T:
#   training targets must be strictly earlier than T.
# ------------------------------------------------------------

# Generate one pre-OOS prediction for December 2018 so that
# turnover for the first OOS month can use actual prior holdings.
prediction_mask = primary["target_month"].between(
    pd.Timestamp("2018-12-31"),
    OOS_END
)

prediction_states = primary.index[prediction_mask]

selector_rows = []
selector_weight_rows = []

for fold_number, state_month in enumerate(
    prediction_states,
    start=1
):
    target_month = primary.loc[
        state_month,
        "target_month"
    ]

    training_mask = (
        primary["target_month"] < target_month
    )

    training_index = primary.index[training_mask]

    X_train_raw = primary.loc[
        training_index,
        FEATURE_COLUMNS
    ].copy()

    X_test_raw = primary.loc[
        [state_month],
        FEATURE_COLUMNS
    ].copy()

    if len(X_train_raw) < 60:
        raise RuntimeError(
            f"Insufficient training rows at {state_month}: "
            f"{len(X_train_raw)}"
        )

    # Fit normalization only on this fold's training data.
    fold_mean = X_train_raw.mean()
    fold_std = X_train_raw.std(ddof=0)

    if fold_std.eq(0).any():
        bad = fold_std.index[fold_std.eq(0)].tolist()
        raise RuntimeError(
            f"Zero-variance fold features: {bad}"
        )

    X_train = (
        X_train_raw - fold_mean
    ) / fold_std

    X_test = (
        X_test_raw - fold_mean
    ) / fold_std

    predictions = {}
    actual_returns = {}

    for expert_number, expert in enumerate(EXPERTS):
        y_train = expert_gross_returns.loc[
            training_index,
            expert
        ]

        model = GradientBoostingRegressor(
            random_state=(
                MODEL_RANDOM_SEED + expert_number
            ),
            **GBM_PARAMETERS
        )

        model.fit(X_train, y_train)

        predictions[expert] = float(
            model.predict(X_test)[0]
        )

        actual_returns[expert] = float(
            expert_gross_returns.loc[
                state_month,
                expert
            ]
        )

    # Deterministic tie-breaking uses EXPERTS list order.
    selected_expert = max(
        EXPERTS,
        key=lambda expert: (
            predictions[expert],
            -EXPERTS.index(expert)
        )
    )

    actual_best_expert = max(
        EXPERTS,
        key=lambda expert: (
            actual_returns[expert],
            -EXPERTS.index(expert)
        )
    )

    selected_weights = expert_weights[
        selected_expert
    ].loc[state_month]

    selector_rows.append(
        {
            "fold_number": fold_number,
            "state_month": state_month,
            "target_month": target_month,
            "training_rows": len(training_index),
            "training_first_target": primary.loc[
                training_index,
                "target_month"
            ].min(),
            "training_last_target": primary.loc[
                training_index,
                "target_month"
            ].max(),
            "selected_expert": selected_expert,
            "actual_best_expert": actual_best_expert,
            "selected_expert_hit": (
                selected_expert == actual_best_expert
            ),
            "selected_actual_gross_return":
                actual_returns[selected_expert],
            "oracle_actual_gross_return":
                actual_returns[actual_best_expert],
            "oracle_gap": (
                actual_returns[actual_best_expert]
                - actual_returns[selected_expert]
            ),
            **{
                f"predicted__{expert}":
                    predictions[expert]
                for expert in EXPERTS
            },
            **{
                f"actual__{expert}":
                    actual_returns[expert]
                for expert in EXPERTS
            },
        }
    )

    selector_weight_rows.append(
        {
            "state_month": state_month,
            "target_month": target_month,
            "selected_expert": selected_expert,
            **{
                asset: selected_weights[asset]
                for asset in PORTFOLIO_ASSETS
            },
        }
    )

selector_predictions = pd.DataFrame(
    selector_rows
).set_index("state_month").sort_index()

selector_weights = pd.DataFrame(
    selector_weight_rows
).set_index("state_month").sort_index()

selector_predictions.to_parquet(
    OUTDIR / "selector_expanding_predictions.parquet"
)

selector_weights.to_parquet(
    OUTDIR / "selector_expanding_weights.parquet"
)

# ------------------------------------------------------------
# 4. Selector drift-adjusted turnover
# ------------------------------------------------------------

cash_by_month = pd.read_parquet(
    OUTDIR / "crsp_30day_tbill_monthly_return.parquet"
).iloc[:, 0]

cash_by_month.index = pd.to_datetime(
    cash_by_month.index
)

selector_state_returns = raw_returns[
    ASSETS
].reindex(selector_weights.index).copy()

selector_state_returns["CASH"] = (
    cash_by_month.reindex(
        selector_weights.index
    )
)

if selector_state_returns.isna().any().any():
    raise RuntimeError(
        "Missing state returns for selector turnover."
    )

def calculate_turnover(weights, state_returns):
    turnover = pd.Series(
        np.nan,
        index=weights.index,
        name="turnover"
    )

    turnover.iloc[0] = 0.0

    for position in range(1, len(weights)):
        current_state = weights.index[position]
        previous_state = weights.index[position - 1]

        previous_weights = weights.loc[
            previous_state,
            PORTFOLIO_ASSETS
        ]

        current_weights = weights.loc[
            current_state,
            PORTFOLIO_ASSETS
        ]

        realized_returns = state_returns.loc[
            current_state,
            PORTFOLIO_ASSETS
        ]

        end_values = (
            previous_weights
            * (1.0 + realized_returns)
        )

        drifted_previous = (
            end_values / end_values.sum()
        )

        turnover.loc[current_state] = (
            0.5
            * (
                current_weights
                - drifted_previous
            ).abs().sum()
        )

    return turnover

selector_turnover = calculate_turnover(
    selector_weights,
    selector_state_returns
)

selector_target_returns = target_returns.reindex(
    selector_weights.index
)

selector_gross_return = (
    selector_weights[PORTFOLIO_ASSETS]
    * selector_target_returns[PORTFOLIO_ASSETS]
).sum(axis=1)

selector_net_returns = {}

for cost_bps in COST_GRID_BPS:
    selector_net_returns[cost_bps] = (
        selector_gross_return
        - (
            cost_bps / 10_000.0
            * selector_turnover
        )
    )

# ------------------------------------------------------------
# 5. OOS selector diagnostics
# ------------------------------------------------------------

selector_oos_mask = (
    selector_predictions["target_month"]
    >= OOS_START
)

selector_oos_index = selector_predictions.index[
    selector_oos_mask
]

if len(selector_oos_index) != 72:
    raise RuntimeError(
        f"Expected 72 selector OOS months; "
        f"found {len(selector_oos_index)}"
    )

selection_frequency = (
    selector_predictions.loc[
        selector_oos_index,
        "selected_expert"
    ]
    .value_counts()
    .reindex(EXPERTS, fill_value=0)
    .rename("selected_months")
    .to_frame()
)

selection_frequency["selection_fraction"] = (
    selection_frequency["selected_months"]
    / len(selector_oos_index)
)

selection_frequency.to_csv(
    OUTDIR / "selector_expert_frequency.csv"
)

cross_sectional_ic_rows = []

for state_month in selector_oos_index:
    predicted = np.array(
        [
            selector_predictions.loc[
                state_month,
                f"predicted__{expert}"
            ]
            for expert in EXPERTS
        ]
    )

    actual = np.array(
        [
            selector_predictions.loc[
                state_month,
                f"actual__{expert}"
            ]
            for expert in EXPERTS
        ]
    )

    correlation = spearmanr(
        predicted,
        actual
    ).statistic

    cross_sectional_ic_rows.append(
        {
            "state_month": state_month,
            "target_month": selector_predictions.loc[
                state_month,
                "target_month"
            ],
            "cross_sectional_spearman_ic": correlation,
        }
    )

cross_sectional_ic = pd.DataFrame(
    cross_sectional_ic_rows
)

cross_sectional_ic.to_csv(
    OUTDIR / "selector_cross_sectional_ic.csv",
    index=False
)

diagnostic_summary = pd.DataFrame(
    [
        {
            "oos_months": len(selector_oos_index),
            "actual_best_expert_hit_rate": (
                selector_predictions.loc[
                    selector_oos_index,
                    "selected_expert_hit"
                ].mean()
            ),
            "mean_oracle_gap_monthly": (
                selector_predictions.loc[
                    selector_oos_index,
                    "oracle_gap"
                ].mean()
            ),
            "median_oracle_gap_monthly": (
                selector_predictions.loc[
                    selector_oos_index,
                    "oracle_gap"
                ].median()
            ),
            "mean_cross_sectional_spearman_ic": (
                cross_sectional_ic[
                    "cross_sectional_spearman_ic"
                ].mean()
            ),
            "median_cross_sectional_spearman_ic": (
                cross_sectional_ic[
                    "cross_sectional_spearman_ic"
                ].median()
            ),
            "average_monthly_turnover": (
                selector_turnover.loc[
                    selector_oos_index
                ].mean()
            ),
            "annualized_turnover": (
                12.0
                * selector_turnover.loc[
                    selector_oos_index
                ].mean()
            ),
        }
    ]
)

diagnostic_summary.to_csv(
    OUTDIR / "selector_diagnostic_summary.csv",
    index=False
)

print("\n" + "=" * 115)
print("A. SELECTOR DIAGNOSTICS")
print("=" * 115)
display(diagnostic_summary)

print("\nExpert selection frequency:")
display(selection_frequency)

# ------------------------------------------------------------
# 6. Performance functions
# ------------------------------------------------------------

def maximum_drawdown(monthly_returns):
    values = pd.Series(
        monthly_returns,
        dtype=float
    ).dropna().to_numpy()

    wealth = np.concatenate(
        [
            np.array([1.0]),
            np.cumprod(1.0 + values)
        ]
    )

    running_peak = np.maximum.accumulate(wealth)

    return float(
        np.min(
            wealth / running_peak - 1.0
        )
    )

def excess_sharpe(monthly_returns, cash_returns):
    r = np.asarray(monthly_returns, dtype=float)
    rf = np.asarray(cash_returns, dtype=float)

    excess = r - rf
    volatility = np.std(excess, ddof=1)

    if volatility <= 0:
        return np.nan

    return float(
        np.sqrt(12.0)
        * np.mean(excess)
        / volatility
    )

def performance_metrics(
    monthly_returns,
    cash_returns,
    turnover=None
):
    r = pd.Series(
        monthly_returns,
        dtype=float
    ).dropna()

    rf = pd.Series(
        cash_returns,
        dtype=float
    ).reindex(r.index)

    if rf.isna().any():
        raise RuntimeError(
            "Cash missing in performance calculation."
        )

    months = len(r)
    ending_wealth = np.prod(1.0 + r.to_numpy())

    cagr = (
        ending_wealth ** (12.0 / months)
        - 1.0
    )

    max_dd = maximum_drawdown(r)

    if turnover is None:
        average_turnover = 0.0
    else:
        average_turnover = (
            pd.Series(turnover)
            .reindex(r.index)
            .mean()
        )

    return {
        "months": months,
        "cagr": cagr,
        "arithmetic_annual_return":
            12.0 * r.mean(),
        "annual_volatility":
            np.sqrt(12.0) * r.std(ddof=1),
        "excess_sharpe":
            excess_sharpe(r, rf),
        "maximum_drawdown": max_dd,
        "calmar": (
            cagr / abs(max_dd)
            if max_dd < 0 else np.nan
        ),
        "total_return":
            ending_wealth - 1.0,
        "average_monthly_turnover":
            average_turnover,
        "annualized_turnover":
            12.0 * average_turnover,
    }

# ------------------------------------------------------------
# 7. Combine selector and benchmark returns
# ------------------------------------------------------------

oos_target_months = primary.loc[
    selector_oos_index,
    "target_month"
]

oos_cash = primary.loc[
    selector_oos_index,
    "target_next__cash"
]

comparison_returns = pd.DataFrame(
    index=selector_oos_index
)

comparison_returns["target_month"] = oos_target_months
comparison_returns["cash"] = oos_cash

for expert in EXPERTS:
    comparison_returns[expert] = (
        baseline_monthly.loc[
            selector_oos_index,
            f"{expert}__net_10bps"
        ]
    )

comparison_returns["GSG_buy_and_hold"] = (
    baseline_monthly.loc[
        selector_oos_index,
        "GSG_buy_and_hold"
    ]
)

comparison_returns["DBC_buy_and_hold"] = (
    baseline_monthly.loc[
        selector_oos_index,
        "DBC_buy_and_hold"
    ]
)

comparison_returns["supervised_selector"] = (
    selector_net_returns[10].loc[
        selector_oos_index
    ]
)

comparison_returns.to_parquet(
    OUTDIR / "selector_and_benchmark_oos_returns.parquet"
)

# ------------------------------------------------------------
# 8. Selector performance and cost sensitivity
# ------------------------------------------------------------

selector_performance_rows = []

for cost_bps in COST_GRID_BPS:
    selector_metrics = performance_metrics(
        selector_net_returns[cost_bps].loc[
            selector_oos_index
        ],
        oos_cash,
        selector_turnover.loc[
            selector_oos_index
        ]
    )

    selector_metrics.update(
        {
            "strategy": "supervised_selector",
            "cost_bps": cost_bps,
        }
    )

    selector_performance_rows.append(
        selector_metrics
    )

for expert in EXPERTS:
    expert_metrics = performance_metrics(
        comparison_returns[expert],
        oos_cash,
        baseline_monthly.loc[
            selector_oos_index,
            f"{expert}__turnover"
        ]
    )

    expert_metrics.update(
        {
            "strategy": expert,
            "cost_bps": 10,
        }
    )

    selector_performance_rows.append(
        expert_metrics
    )

for benchmark in [
    "GSG_buy_and_hold",
    "DBC_buy_and_hold",
]:
    benchmark_metrics = performance_metrics(
        comparison_returns[benchmark],
        oos_cash,
        turnover=None
    )

    benchmark_metrics.update(
        {
            "strategy": benchmark,
            "cost_bps": 10,
        }
    )

    selector_performance_rows.append(
        benchmark_metrics
    )

selector_performance = pd.DataFrame(
    selector_performance_rows
).sort_values(
    ["cost_bps", "excess_sharpe"],
    ascending=[True, False]
)

selector_performance.to_csv(
    OUTDIR / "selector_performance_and_costs.csv",
    index=False
)

print("\n" + "=" * 115)
print("B. SELECTOR PERFORMANCE AND PRIMARY COMPARISONS")
print("=" * 115)
display(selector_performance)

# ------------------------------------------------------------
# 9. Circular moving-block bootstrap
# ------------------------------------------------------------

def circular_block_indices(
    observations,
    block_length,
    rng
):
    blocks_needed = math.ceil(
        observations / block_length
    )

    starts = rng.integers(
        0,
        observations,
        size=blocks_needed
    )

    indices = []

    for start in starts:
        block = (
            np.arange(
                start,
                start + block_length
            )
            % observations
        )
        indices.extend(block.tolist())

    return np.asarray(
        indices[:observations],
        dtype=int
    )

def pairwise_block_bootstrap(
    candidate,
    benchmark,
    cash,
    block_length,
    replications,
    seed
):
    candidate = np.asarray(candidate, dtype=float)
    benchmark = np.asarray(benchmark, dtype=float)
    cash = np.asarray(cash, dtype=float)

    difference = candidate - benchmark
    observations = len(difference)

    observed_mean = difference.mean()

    observed_sharpe_difference = (
        excess_sharpe(candidate, cash)
        - excess_sharpe(benchmark, cash)
    )

    centered_difference = (
        difference - observed_mean
    )

    rng = np.random.default_rng(seed)

    boot_mean = np.empty(replications)
    boot_centered_mean = np.empty(replications)
    boot_sharpe_difference = np.empty(
        replications
    )

    for replication in range(replications):
        indices = circular_block_indices(
            observations,
            block_length,
            rng
        )

        boot_mean[replication] = (
            difference[indices].mean()
        )

        boot_centered_mean[replication] = (
            centered_difference[indices].mean()
        )

        boot_sharpe_difference[replication] = (
            excess_sharpe(
                candidate[indices],
                cash[indices]
            )
            - excess_sharpe(
                benchmark[indices],
                cash[indices]
            )
        )

    p_value = (
        1.0
        + np.sum(
            np.abs(boot_centered_mean)
            >= abs(observed_mean)
        )
    ) / (replications + 1.0)

    return {
        "annualized_mean_difference":
            12.0 * observed_mean,
        "mean_difference_ci_2_5":
            12.0 * np.percentile(
                boot_mean,
                2.5
            ),
        "mean_difference_ci_97_5":
            12.0 * np.percentile(
                boot_mean,
                97.5
            ),
        "two_sided_centered_p_value":
            p_value,
        "observed_sharpe_difference":
            observed_sharpe_difference,
        "sharpe_difference_ci_2_5":
            np.percentile(
                boot_sharpe_difference,
                2.5
            ),
        "sharpe_difference_ci_97_5":
            np.percentile(
                boot_sharpe_difference,
                97.5
            ),
    }

comparison_strategies = [
    "momentum_6m_top2_invvol",
    "inverse_vol_12m",
    "equal_weight",
    "momentum_12m_top2_invvol",
    "absolute_momentum_12m_cash",
    "DBC_buy_and_hold",
    "GSG_buy_and_hold",
]

bootstrap_rows = []

for block_length in BOOTSTRAP_BLOCK_LENGTHS:
    for benchmark in comparison_strategies:
        result = pairwise_block_bootstrap(
            comparison_returns[
                "supervised_selector"
            ],
            comparison_returns[benchmark],
            comparison_returns["cash"],
            block_length=block_length,
            replications=BOOTSTRAP_REPLICATIONS,
            seed=(
                BOOTSTRAP_SEED
                + block_length
                + comparison_strategies.index(
                    benchmark
                )
            )
        )

        result.update(
            {
                "candidate": "supervised_selector",
                "benchmark": benchmark,
                "block_length": block_length,
                "bootstrap_replications":
                    BOOTSTRAP_REPLICATIONS,
            }
        )

        bootstrap_rows.append(result)

bootstrap_results = pd.DataFrame(bootstrap_rows)

# ------------------------------------------------------------
# 10. Holm multiple-comparison correction
# ------------------------------------------------------------

def holm_adjust(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float
    )

    count = len(p_values)
    order = np.argsort(p_values)

    adjusted_sorted = np.empty(count)
    running_maximum = 0.0

    for rank, original_position in enumerate(order):
        adjusted = (
            (count - rank)
            * p_values[original_position]
        )

        running_maximum = max(
            running_maximum,
            adjusted
        )

        adjusted_sorted[rank] = min(
            running_maximum,
            1.0
        )

    adjusted = np.empty(count)

    for rank, original_position in enumerate(order):
        adjusted[original_position] = (
            adjusted_sorted[rank]
        )

    return adjusted

primary_block_mask = (
    bootstrap_results["block_length"].eq(6)
)

bootstrap_results.loc[
    primary_block_mask,
    "holm_adjusted_p_value"
] = holm_adjust(
    bootstrap_results.loc[
        primary_block_mask,
        "two_sided_centered_p_value"
    ].to_numpy()
)

bootstrap_results.to_csv(
    OUTDIR / "selector_pairwise_block_bootstrap.csv",
    index=False
)

print("\n" + "=" * 115)
print("C. PAIRWISE BOOTSTRAP — PRIMARY SIX-MONTH BLOCK")
print("=" * 115)
display(
    bootstrap_results.loc[
        bootstrap_results["block_length"].eq(6)
    ].sort_values("benchmark")
)

# ------------------------------------------------------------
# 11. Centered moving-block max-mean reality check
#
# Candidate set is frozen and reported explicitly.
# Benchmark is the strongest transparent strategy:
# momentum_6m_top2_invvol.
# ------------------------------------------------------------

reality_benchmark = "momentum_6m_top2_invvol"

reality_candidates = [
    "supervised_selector",
    "inverse_vol_12m",
    "equal_weight",
    "momentum_12m_top2_invvol",
    "absolute_momentum_12m_cash",
    "DBC_buy_and_hold",
    "GSG_buy_and_hold",
]

reality_differences = pd.DataFrame(
    {
        candidate: (
            comparison_returns[candidate]
            - comparison_returns[
                reality_benchmark
            ]
        )
        for candidate in reality_candidates
    }
)

observed_candidate_means = (
    reality_differences.mean()
)

observed_max_mean = (
    observed_candidate_means.max()
)

observed_best_candidate = (
    observed_candidate_means.idxmax()
)

reality_rows = []

for block_length in BOOTSTRAP_BLOCK_LENGTHS:
    centered = (
        reality_differences
        - reality_differences.mean()
    )

    rng = np.random.default_rng(
        BOOTSTRAP_SEED + 1000 + block_length
    )

    bootstrap_max_means = np.empty(
        BOOTSTRAP_REPLICATIONS
    )

    for replication in range(
        BOOTSTRAP_REPLICATIONS
    ):
        indices = circular_block_indices(
            len(centered),
            block_length,
            rng
        )

        bootstrap_max_means[replication] = (
            centered.iloc[indices]
            .mean()
            .max()
        )

    reality_p_value = (
        1.0
        + np.sum(
            bootstrap_max_means
            >= observed_max_mean
        )
    ) / (BOOTSTRAP_REPLICATIONS + 1.0)

    reality_rows.append(
        {
            "benchmark":
                reality_benchmark,
            "candidate_count":
                len(reality_candidates),
            "candidate_set":
                " | ".join(reality_candidates),
            "observed_best_candidate":
                observed_best_candidate,
            "observed_max_annualized_mean_difference":
                12.0 * observed_max_mean,
            "block_length":
                block_length,
            "bootstrap_replications":
                BOOTSTRAP_REPLICATIONS,
            "centered_max_mean_p_value":
                reality_p_value,
        }
    )

reality_check = pd.DataFrame(reality_rows)

reality_check.to_csv(
    OUTDIR / "centered_max_mean_reality_check.csv",
    index=False
)

print("\n" + "=" * 115)
print("D. MULTIPLE-MODEL REALITY CHECK")
print("=" * 115)
display(reality_check)

# ------------------------------------------------------------
# 12. Period-exclusion robustness
# ------------------------------------------------------------

period_definitions = {
    "Full OOS 2019-2024": (
        comparison_returns["target_month"]
        .between(
            pd.Timestamp("2019-01-31"),
            pd.Timestamp("2024-12-31")
        )
    ),
    "Pre-COVID 2019": (
        comparison_returns["target_month"].dt.year
        .eq(2019)
    ),
    "COVID/reflation 2020-2021": (
        comparison_returns["target_month"].dt.year
        .isin([2020, 2021])
    ),
    "Post-2021 2022-2024": (
        comparison_returns["target_month"].dt.year
        .isin([2022, 2023, 2024])
    ),
    "Excluding 2020-2021": (
        ~comparison_returns["target_month"].dt.year
        .isin([2020, 2021])
    ),
}

robustness_strategies = [
    "supervised_selector",
    "momentum_6m_top2_invvol",
    "inverse_vol_12m",
    "equal_weight",
    "DBC_buy_and_hold",
    "GSG_buy_and_hold",
]

period_rows = []

for period_name, mask in period_definitions.items():
    period_index = comparison_returns.index[mask]

    for strategy in robustness_strategies:
        metrics = performance_metrics(
            comparison_returns.loc[
                period_index,
                strategy
            ],
            comparison_returns.loc[
                period_index,
                "cash"
            ],
            turnover=(
                selector_turnover.loc[
                    period_index
                ]
                if strategy
                == "supervised_selector"
                else None
            )
        )

        metrics.update(
            {
                "period": period_name,
                "strategy": strategy,
            }
        )

        period_rows.append(metrics)

period_robustness = pd.DataFrame(period_rows)

period_robustness.to_csv(
    OUTDIR / "selector_period_robustness.csv",
    index=False
)

print("\n" + "=" * 115)
print("E. PERIOD-EXCLUSION ROBUSTNESS")
print("=" * 115)
display(period_robustness)

# ------------------------------------------------------------
# 13. Annual excess contribution and leave-one-year-out
# ------------------------------------------------------------

comparison_returns[
    "selector_minus_momentum6"
] = (
    comparison_returns["supervised_selector"]
    - comparison_returns[
        "momentum_6m_top2_invvol"
    ]
)

annual_contribution = (
    comparison_returns.groupby(
        comparison_returns["target_month"].dt.year
    )["selector_minus_momentum6"]
    .agg(
        months="size",
        summed_monthly_difference="sum",
        mean_monthly_difference="mean"
    )
    .reset_index()
    .rename(columns={"target_month": "year"})
)

annual_contribution[
    "annualized_mean_difference"
] = (
    12.0
    * annual_contribution[
        "mean_monthly_difference"
    ]
)

annual_contribution.to_csv(
    OUTDIR / "selector_annual_excess_contribution.csv",
    index=False
)

leave_one_year_out_rows = []

years = sorted(
    comparison_returns["target_month"]
    .dt.year
    .unique()
)

for excluded_year in years:
    keep = (
        comparison_returns["target_month"]
        .dt.year.ne(excluded_year)
    )

    kept_index = comparison_returns.index[keep]

    for strategy in [
        "supervised_selector",
        "momentum_6m_top2_invvol",
    ]:
        metrics = performance_metrics(
            comparison_returns.loc[
                kept_index,
                strategy
            ],
            comparison_returns.loc[
                kept_index,
                "cash"
            ],
            turnover=(
                selector_turnover.loc[
                    kept_index
                ]
                if strategy
                == "supervised_selector"
                else None
            )
        )

        metrics.update(
            {
                "excluded_year":
                    excluded_year,
                "strategy":
                    strategy,
            }
        )

        leave_one_year_out_rows.append(metrics)

leave_one_year_out = pd.DataFrame(
    leave_one_year_out_rows
)

leave_one_year_out.to_csv(
    OUTDIR / "selector_leave_one_year_out.csv",
    index=False
)

print("\n" + "=" * 115)
print("F. ANNUAL SELECTOR MINUS MOMENTUM-6 CONTRIBUTION")
print("=" * 115)
display(annual_contribution)

print("\nLeave-one-year-out results:")
display(leave_one_year_out)

# ------------------------------------------------------------
# 14. Final audit summary
# ------------------------------------------------------------

selector_primary_row = (
    selector_performance.loc[
        selector_performance["strategy"]
        .eq("supervised_selector")
        & selector_performance["cost_bps"]
        .eq(10)
    ]
    .iloc[0]
)

momentum_primary_row = (
    selector_performance.loc[
        selector_performance["strategy"]
        .eq("momentum_6m_top2_invvol")
    ]
    .iloc[0]
)

momentum_bootstrap_row = (
    bootstrap_results.loc[
        bootstrap_results["benchmark"]
        .eq("momentum_6m_top2_invvol")
        & bootstrap_results["block_length"]
        .eq(6)
    ]
    .iloc[0]
)

final_summary = pd.DataFrame(
    [
        {
            "selector_cagr":
                selector_primary_row["cagr"],
            "selector_excess_sharpe":
                selector_primary_row["excess_sharpe"],
            "momentum6_cagr":
                momentum_primary_row["cagr"],
            "momentum6_excess_sharpe":
                momentum_primary_row["excess_sharpe"],
            "selector_minus_momentum6_annualized_mean":
                momentum_bootstrap_row[
                    "annualized_mean_difference"
                ],
            "selector_vs_momentum6_p_value":
                momentum_bootstrap_row[
                    "two_sided_centered_p_value"
                ],
            "selector_vs_momentum6_holm_p_value":
                momentum_bootstrap_row[
                    "holm_adjusted_p_value"
                ],
            "selector_beats_momentum6_point_estimate":
                (
                    selector_primary_row["cagr"]
                    > momentum_primary_row["cagr"]
                ),
            "selector_statistically_beats_momentum6_at_5pct":
                (
                    momentum_bootstrap_row[
                        "two_sided_centered_p_value"
                    ] < 0.05
                    and momentum_bootstrap_row[
                        "annualized_mean_difference"
                    ] > 0
                ),
        }
    ]
)

final_summary.to_csv(
    OUTDIR / "selector_final_decision_summary.csv",
    index=False
)

print("\n" + "=" * 115)
print("G. SELECTOR DECISION SUMMARY")
print("=" * 115)
display(final_summary)

# ------------------------------------------------------------
# 15. Hash outputs
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()

hash_rows = []

for path in sorted(OUTDIR.iterdir()):
    if (
        path.is_file()
        and path.name
        != "audit_file_hashes_stage5.csv"
    ):
        hash_rows.append(
            {
                "filename": path.name,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

pd.DataFrame(hash_rows).to_csv(
    OUTDIR / "audit_file_hashes_stage5.csv",
    index=False
)

print("\n" + "=" * 115)
print("AUDIT 5 COMPLETE")
print("=" * 115)
print("Model: fixed supervised Gradient Boosting expert selector")
print("Walk-forward: expanding monthly")
print("Hyperparameter search: NONE")
print("Primary benchmark: momentum_6m_top2_invvol")
print("Bootstrap replications:", BOOTSTRAP_REPLICATIONS)
print("Output directory:", OUTDIR)
print()
print("PASTE BACK SECTIONS A, B, C, D, E, F, AND G.")